In [115]:
import warnings
warnings.filterwarnings('ignore')

import os
import pickle

import numpy as np
import pandas as pd

from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta

from pandas.tseries.offsets import MonthEnd, MonthBegin
from maricovault.MaricoDB import MaricoSnowflake

from joblib import Parallel, delayed

In [116]:
def get_dbconnection(db_name): 

    KEY_VAULT_NAME = "prod-pwd"
    if db_name == 'PROD':
        db_name = 'prod'
    else:
        db_name = 'dev'

    msf = MaricoSnowflake(KEY_VAULT_NAME)
    msf.get_db_credentials(db_name=db_name)
    msf.connect()
    dbconnection = msf.get_connection()
    
    return dbconnection

In [117]:
dev_conn = get_dbconnection('DEV')
prod_conn = get_dbconnection('PROD')


Credentials retrieved successfully for dev db.

Credentials retrieved successfully for prod db.


### Helper Functions

In [118]:
realignment_df = pd.read_sql(
    """select * from trn_mil_asm_psku_realignment""",
    dev_conn
)
realignment_df.columns = realignment_df.columns.str.lower()

def demand_driver_realign_pskus(data, channel):
    """
    Realign the old pskus to new pskus and return updated data.

    Args:
        data: pandas dataframe
        - master dataframe having all the pskus
    
    Return:
        data: pandas dataframe
        - dataframe 
    """
    realignment_data = realignment_df.copy()
    realignment_data.columns = realignment_data.columns.str.lower()
    realignment_data = realignment_data[
        (realignment_data["channel"] == channel)
        | (realignment_data["channel"] == channel + " B2C")
        | (realignment_data["channel"] == "ALL")
    ]

    data["parent_material_code"] = data["parent_material_code"].astype(int)

    for grp, grp_data in realignment_data.groupby(by=["psku old", "asm"]):
        old_psku, old_asm = grp
        new_psku = grp_data["psku new"].values[0]
        if old_asm != "ALL":
            condition = (data["parent_material_code"] == old_psku) & (
                data["asm_area_code"] == old_asm
            )
        else:
            condition = data["parent_material_code"] == old_psku

        data.loc[condition, "parent_material_code"] = new_psku

    return data

### Aggregation

In [119]:
data_query = f"""
    select * from TRN_MIL_DF_HEURISTICS_OUTPUT 
"""
df_heuristics = pd.read_sql(data_query, dev_conn)
df_heuristics

,CHANNEL,PORTFOLIO,BRAND,BRAND CLASS,RUN_MONTH,M MONTH,MONTH,ASM,DEPOT,PSKU,PROPHET VOL,RF_VOL,PROPHET HEURISTIC VOL,RF HEURISTIC VOL
0,ECOM,Hair Oils,ADV-AHO-R,B,2025-11-30,M,2025-11-30,BCE1,D231,718589,5.264959,2.700,5.264959,4.500
1,ECOM,Hair Oils,ADV-AHO-R,B,2025-11-30,M+1,2025-12-31,BCE1,D231,718589,4.547336,3.600,4.547336,4.500
2,ECOM,Hair Oils,ADV-AHO-R,B,2025-11-30,M+2,2026-01-31,BCE1,D231,718589,7.053803,2.700,7.053803,4.500
3,ECOM,Hair Oils,ADV-AHO-R,B,2025-11-30,M+3,2026-02-28,BCE1,D231,718589,5.662848,1.800,5.662848,4.500
4,ECOM,Hair Oils,ADV-AHO-R,B,2025-11-30,M+4,2026-03-31,BCE1,D231,718589,0.000000,2.700,4.500000,4.500
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
282199,QCOM,Male Grooming,SW_HR_WAX,C,2025-10-31,M+1,2025-11-30,QCW2,D461,718836,3.767871,4.908,5.040000,5.040
282200,QCOM,Male Grooming,SW_HR_WAX,C,2025-10-31,M+2,2025-12-31,QCW2,D461,718836,4.907447,4.932,5.040000,5.040
282201,QCOM,Male Grooming,SW_HR_WAX,C,2025-10-31,M+3,2026-01-31,QCW2,D461,718836,4.751205,4.260,5.040000,5.040
282202,QCOM,Male Grooming,SW_HR_WAX,C,2025-10-31,M+4,2026-02-28,QCW2,D461,718836,8.038001,4.776,8.038001,5.040


In [120]:
df_heuristics.columns = df_heuristics.columns.str.lower()
df_heuristics['run_month'] = pd.to_datetime(df_heuristics['run_month'])
df_heuristics['month'] = pd.to_datetime(df_heuristics['month'])
df_heuristics


,channel,portfolio,brand,brand class,run_month,m month,month,asm,depot,psku,prophet vol,rf_vol,prophet heuristic vol,rf heuristic vol
0,ECOM,Hair Oils,ADV-AHO-R,B,2025-11-30,M,2025-11-30,BCE1,D231,718589,5.264959,2.700,5.264959,4.500
1,ECOM,Hair Oils,ADV-AHO-R,B,2025-11-30,M+1,2025-12-31,BCE1,D231,718589,4.547336,3.600,4.547336,4.500
2,ECOM,Hair Oils,ADV-AHO-R,B,2025-11-30,M+2,2026-01-31,BCE1,D231,718589,7.053803,2.700,7.053803,4.500
3,ECOM,Hair Oils,ADV-AHO-R,B,2025-11-30,M+3,2026-02-28,BCE1,D231,718589,5.662848,1.800,5.662848,4.500
4,ECOM,Hair Oils,ADV-AHO-R,B,2025-11-30,M+4,2026-03-31,BCE1,D231,718589,0.000000,2.700,4.500000,4.500
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
282199,QCOM,Male Grooming,SW_HR_WAX,C,2025-10-31,M+1,2025-11-30,QCW2,D461,718836,3.767871,4.908,5.040000,5.040
282200,QCOM,Male Grooming,SW_HR_WAX,C,2025-10-31,M+2,2025-12-31,QCW2,D461,718836,4.907447,4.932,5.040000,5.040
282201,QCOM,Male Grooming,SW_HR_WAX,C,2025-10-31,M+3,2026-01-31,QCW2,D461,718836,4.751205,4.260,5.040000,5.040
282202,QCOM,Male Grooming,SW_HR_WAX,C,2025-10-31,M+4,2026-02-28,QCW2,D461,718836,8.038001,4.776,8.038001,5.040


In [121]:
# delete_query = """
# DELETE FROM TRN_MIL_DF_SHARED
# WHERE "run_month" IN ('2025-12-31', '2026-01-31')
# """

# cur = dev_conn.cursor()
# cur.execute(delete_query)
# cur.close()


In [122]:
data_query = """
SELECT *
FROM TRN_MIL_DF_SHARED
   
"""

df = pd.read_sql(data_query, dev_conn)

In [123]:
df.columns = df.columns.str.lower()
df['run_month'] = pd.to_datetime(df['run_month'])
df['month'] = pd.to_datetime(df['month'])
df

,channel,portfolio,brand,brand class,run_month,m month,month,asm,depot,psku,pred vol (roum)
0,GT,Hair Oils,H&C,B,2025-11-30,M,2025-11-30,AURG,D3A4,718316,85.032500
1,GT,Hair Oils,H&C,B,2025-11-30,M+1,2025-12-31,AURG,D3A4,718316,85.032500
2,GT,Hair Oils,H&C,B,2025-11-30,M+2,2026-01-31,AURG,D3A4,718316,85.032500
3,GT,Hair Oils,H&C,B,2025-11-30,M+3,2026-02-28,AURG,D3A4,718316,85.032500
4,GT,Hair Oils,H&C,B,2025-11-30,M+4,2026-03-31,AURG,D3A4,718316,85.032500
...,...,...,...,...,...,...,...,...,...,...,...
652137,MT,Foods,TRU_ELMNT,NPD,2026-05-31,M+4,2026-09-30,MCW2,D463,811219,0.015333
652138,MT,Foods,TRU_ELMNT,NPD,2026-05-31,M+1,2026-06-30,MCW2,D463,811220,0.023000
652139,MT,Foods,TRU_ELMNT,NPD,2026-05-31,M+2,2026-07-31,MCW2,D463,811220,0.023000
652140,MT,Foods,TRU_ELMNT,NPD,2026-05-31,M+3,2026-08-31,MCW2,D463,811220,0.023000


In [124]:
df[(df['month'] == '2026-04-30') & (df['channel']=='GT') & (df['run_month']=='2026-03-31')]['pred vol (roum)'].sum()


5346018.0036901245

In [125]:
# # make sure run_month is datetime
# df["run_month"] = pd.to_datetime(df["run_month"])
# upload_df["run_month"] = pd.to_datetime(upload_df["run_month"])

# # filter condition
# mask = (df["channel"] == "GT") & (df["run_month"] == "2026-01-31")

# # remove those rows
# df_filtered = df.loc[~mask]

# # concat upload_df
# final_df = pd.concat([df_filtered, upload_df], ignore_index=True)


In [126]:
df_heuristics.columns

Index(['channel', 'portfolio', 'brand', 'brand class', 'run_month', 'm month',
       'month', 'asm', 'depot', 'psku', 'prophet vol', 'rf_vol',
       'prophet heuristic vol', 'rf heuristic vol'],
      dtype='object')

In [127]:
df = df.merge(df_heuristics[['channel','asm', 'depot', 'psku', 'run_month', 'month','prophet vol', 'rf_vol',
       'prophet heuristic vol', 'rf heuristic vol']], on = ['channel','asm', 'depot', 'psku', 'run_month', 'month'],
       how = 'left')

In [128]:
df.isnull().sum()

channel                       0
portfolio                     0
brand                         0
brand class                 422
run_month                     0
m month                       0
month                         0
asm                           0
depot                         0
psku                          0
pred vol (roum)               0
prophet vol              446363
rf_vol                   446363
prophet heuristic vol    410083
rf heuristic vol         410083
dtype: int64

In [129]:
# df = df[df['Month'] == '2025-12-01']
# df

In [130]:
import numpy as np

df['final_channel'] = np.where(
    df['channel'].isin(['QCOM', 'GT']),
    df['channel'],
    np.where(
        df['channel'].isin(['ECOM', 'MT']) &
        df['asm'].astype(str).str.startswith('B'),
        'B2B',
        df['channel']
    )
)
df

,channel,portfolio,brand,brand class,run_month,m month,month,asm,depot,psku,pred vol (roum),prophet vol,rf_vol,prophet heuristic vol,rf heuristic vol,final_channel
0,GT,Hair Oils,H&C,B,2025-11-30,M,2025-11-30,AURG,D3A4,718316,85.032500,63.839080,26.859371,53.455636,79.363667,GT
1,GT,Hair Oils,H&C,B,2025-11-30,M+1,2025-12-31,AURG,D3A4,718316,85.032500,36.136155,23.712371,36.136155,79.363667,GT
2,GT,Hair Oils,H&C,B,2025-11-30,M+2,2026-01-31,AURG,D3A4,718316,85.032500,54.246739,20.307629,33.559920,79.363667,GT
3,GT,Hair Oils,H&C,B,2025-11-30,M+3,2026-02-28,AURG,D3A4,718316,85.032500,34.656392,7.917514,34.656392,79.363667,GT
4,GT,Hair Oils,H&C,B,2025-11-30,M+4,2026-03-31,AURG,D3A4,718316,85.032500,41.879330,10.762457,0.000000,79.363667,GT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
652137,MT,Foods,TRU_ELMNT,NPD,2026-05-31,M+4,2026-09-30,MCW2,D463,811219,0.015333,NaN,NaN,NaN,NaN,MT
652138,MT,Foods,TRU_ELMNT,NPD,2026-05-31,M+1,2026-06-30,MCW2,D463,811220,0.023000,NaN,NaN,NaN,NaN,MT
652139,MT,Foods,TRU_ELMNT,NPD,2026-05-31,M+2,2026-07-31,MCW2,D463,811220,0.023000,NaN,NaN,NaN,NaN,MT
652140,MT,Foods,TRU_ELMNT,NPD,2026-05-31,M+3,2026-08-31,MCW2,D463,811220,0.023000,NaN,NaN,NaN,NaN,MT


In [131]:
df.columns

Index(['channel', 'portfolio', 'brand', 'brand class', 'run_month', 'm month',
       'month', 'asm', 'depot', 'psku', 'pred vol (roum)', 'prophet vol',
       'rf_vol', 'prophet heuristic vol', 'rf heuristic vol', 'final_channel'],
      dtype='object')

In [132]:
df = df.groupby(['final_channel', 'portfolio', 'brand', 'brand class', 'run_month', 'm month',
       'month', 'asm', 'depot', 'psku'])[['pred vol (roum)', 'prophet vol',
       'rf_vol', 'prophet heuristic vol', 'rf heuristic vol']].sum().reset_index()#['Channel'].unique()
df.rename(columns = {'final_channel':'channel'}, inplace = True)
df

,channel,portfolio,brand,brand class,run_month,m month,month,asm,depot,psku,pred vol (roum),prophet vol,rf_vol,prophet heuristic vol,rf heuristic vol
0,B2B,0,JH_FRG_L,0x2a,2026-05-31,M+1,2026-06-30,BCN2,D117,734910,0.0,0.000000,0.000000,0.0,0.0
1,B2B,0,JH_FRG_L,0x2a,2026-05-31,M+1,2026-06-30,BCN2,D117,734988,0.0,0.000000,0.000000,0.0,0.0
2,B2B,0,JH_FRG_L,0x2a,2026-05-31,M+1,2026-06-30,BCN2,D117,734993,0.0,0.000000,0.000000,0.0,0.0
3,B2B,0,JH_FRG_L,0x2a,2026-05-31,M+1,2026-06-30,BCN2,D117,734995,0.0,0.000000,0.000000,0.0,0.0
4,B2B,0,JH_FRG_L,0x2a,2026-05-31,M+1,2026-06-30,BCN2,D117,735043,0.0,0.000000,0.000000,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
640788,QCOM,Skin Care,PURSNS_ML,C,2025-11-30,M+4,2026-03-31,QCW2,D461,809807,0.0,14.685752,2.232000,0.0,0.0
640789,QCOM,Skin Care,PURSNS_ML,C,2025-11-30,M+4,2026-03-31,QCW2,D461,809850,0.0,0.000000,0.120000,0.0,0.0
640790,QCOM,Skin Care,PURSNS_ML,C,2025-11-30,M+4,2026-03-31,QCW2,D461,809865,0.0,0.000000,0.746667,0.0,0.0
640791,QCOM,Skin Care,PURSNS_ML,C,2025-11-30,M+4,2026-03-31,QCW2,D461,809866,0.0,0.000000,0.000000,0.0,0.0


In [133]:
df[(df['run_month'] == '2026-04-30')]['channel'].unique()

array(['GT', 'MT'], dtype=object)

### Actuals

In [134]:
actuals_query = """
SELECT
    CM.channel_name,
    CM.asm_area_code,
    CM.depot_code,
    MM.parent_material_code,
    MM.material_group_code,
    LAST_DAY(MESR.month_date) AS month_date,
    SUM(pri_actuals_vol_rum) AS pri_actuals_vol_rum,
    SUM(pri_apo_plan_vol_rum) AS pri_apo_plan_vol_rum,
    SUM(MESR.sec_apo_plan_vol_rum) AS sec_apo_plan_vol_rum,
    SUM(MESR.sec_actuals_vol_rum) AS sec_actuals_vol_rum
FROM
    dwh_bpm_dist_sku_daily MESR
JOIN
(
    SELECT
        material_code,
        parent_material_code,
        material_group_code,
        uom_reporting,
        vol_per_unit
    FROM 
        mst_material
    WHERE
        company_code='MIL' AND
        latest_record_ind=1
) MM ON MESR.material_code = MM.material_code
JOIN
(
    SELECT DISTINCT
        channel_name, 
        asm_area_code,
        customer_code,
        depot_code
    FROM
        mst_customer
    WHERE
        company_code='MIL' AND
        latest_record_ind=1
) CM ON MESR.distributor_code = CM.customer_code
WHERE
    month_date BETWEEN '2023-01-31' AND '2026-06-30' AND
    CM.channel_name IN ('MT', 'E-Commerce', 'Q-Commerce', 'GT')
GROUP BY 1, 2, 3, 4, 5, 6
ORDER BY 1, 2, 3, 4, 6
"""

actuals_df = pd.read_sql(
    actuals_query,
    prod_conn
)

In [135]:
actuals_df.columns = actuals_df.columns.str.lower()
actuals_df['month_date'] = pd.to_datetime(actuals_df['month_date'])

In [136]:
actuals_df.head()

,channel_name,asm_area_code,depot_code,parent_material_code,material_group_code,month_date,pri_actuals_vol_rum,pri_apo_plan_vol_rum,sec_apo_plan_vol_rum,sec_actuals_vol_rum
0,E-Commerce,BCE1,D231,718297,PCNO(R),2023-01-31,0.000,0.000,0.808,0.076
1,E-Commerce,BCE1,D231,718297,PCNO(R),2023-02-28,0.229,0.525,0.774,0.229
2,E-Commerce,BCE1,D231,718297,PCNO(R),2023-03-31,0.364,0.549,0.724,0.364
3,E-Commerce,BCE1,D231,718297,PCNO(R),2023-04-30,0.057,0.189,0.282,0.057
4,E-Commerce,BCE1,D231,718297,PCNO(R),2023-05-31,-0.021,0.335,0.528,-0.021


In [137]:
actuals_df.duplicated(subset=['channel_name', 'asm_area_code', 'depot_code', 'parent_material_code','month_date']).sum()

0

In [138]:
# actuals_df['key'] =  actuals_df['asm_area_code'].astype(str) + '_' + actuals_df['depot_code'].astype(str) + '_' + actuals_df['parent_material_code'].astype(str) 
# actuals_df

In [139]:
# actuals_df[(actuals_df['channel_name'] == 'GT') & (actuals_df['month_date']=='2026-02-28')]['key'].nunique()

In [140]:
actuals_df['channel_name'] = actuals_df['channel_name'].replace({
    'E-Commerce': 'ECOM', 'Q-Commerce': 'QCOM'})

In [141]:
actuals_df

,channel_name,asm_area_code,depot_code,parent_material_code,material_group_code,month_date,pri_actuals_vol_rum,pri_apo_plan_vol_rum,sec_apo_plan_vol_rum,sec_actuals_vol_rum
0,ECOM,BCE1,D231,718297,PCNO(R),2023-01-31,0.000,0.000000,0.808,0.076
1,ECOM,BCE1,D231,718297,PCNO(R),2023-02-28,0.229,0.525000,0.774,0.229
2,ECOM,BCE1,D231,718297,PCNO(R),2023-03-31,0.364,0.549000,0.724,0.364
3,ECOM,BCE1,D231,718297,PCNO(R),2023-04-30,0.057,0.189000,0.282,0.057
4,ECOM,BCE1,D231,718297,PCNO(R),2023-05-31,-0.021,0.335000,0.528,-0.021
...,...,...,...,...,...,...,...,...,...,...
2354825,QCOM,QCW2,D463,811279,SAF_CDPRS,2026-05-31,0.000,0.059530,0.000,0.000
2354826,QCOM,QCW2,D463,811279,SAF_CDPRS,2026-06-30,0.000,0.062444,0.000,0.000
2354827,QCOM,QCW2,D463,811287,PA_RSW_SR,2026-04-30,0.000,13.636364,0.000,0.000
2354828,QCOM,QCW2,D463,811287,PA_RSW_SR,2026-05-31,0.000,21.106628,0.000,0.000


In [142]:
import numpy as np

actuals_df['final_channel'] = np.where(
    actuals_df['channel_name'].isin(['QCOM', 'GT']),
    actuals_df['channel_name'],
    np.where(
        actuals_df['channel_name'].isin(['ECOM', 'MT']) &
        actuals_df['asm_area_code'].astype(str).str.startswith('B'),
        'B2B',
        actuals_df['channel_name']
    )
)
actuals_df

,channel_name,asm_area_code,depot_code,parent_material_code,material_group_code,month_date,pri_actuals_vol_rum,pri_apo_plan_vol_rum,sec_apo_plan_vol_rum,sec_actuals_vol_rum,final_channel
0,ECOM,BCE1,D231,718297,PCNO(R),2023-01-31,0.000,0.000000,0.808,0.076,B2B
1,ECOM,BCE1,D231,718297,PCNO(R),2023-02-28,0.229,0.525000,0.774,0.229,B2B
2,ECOM,BCE1,D231,718297,PCNO(R),2023-03-31,0.364,0.549000,0.724,0.364,B2B
3,ECOM,BCE1,D231,718297,PCNO(R),2023-04-30,0.057,0.189000,0.282,0.057,B2B
4,ECOM,BCE1,D231,718297,PCNO(R),2023-05-31,-0.021,0.335000,0.528,-0.021,B2B
...,...,...,...,...,...,...,...,...,...,...,...
2354825,QCOM,QCW2,D463,811279,SAF_CDPRS,2026-05-31,0.000,0.059530,0.000,0.000,QCOM
2354826,QCOM,QCW2,D463,811279,SAF_CDPRS,2026-06-30,0.000,0.062444,0.000,0.000,QCOM
2354827,QCOM,QCW2,D463,811287,PA_RSW_SR,2026-04-30,0.000,13.636364,0.000,0.000,QCOM
2354828,QCOM,QCW2,D463,811287,PA_RSW_SR,2026-05-31,0.000,21.106628,0.000,0.000,QCOM


In [143]:
actuals_df = actuals_df.groupby(['final_channel', 'asm_area_code', 'depot_code', 'parent_material_code',
       'material_group_code', 'month_date'])[['pri_actuals_vol_rum',
       'pri_apo_plan_vol_rum', 'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum']].sum().reset_index()
actuals_df

,final_channel,asm_area_code,depot_code,parent_material_code,material_group_code,month_date,pri_actuals_vol_rum,pri_apo_plan_vol_rum,sec_apo_plan_vol_rum,sec_actuals_vol_rum
0,B2B,BCE1,D231,702478,PADV-HRCR,2024-09-30,0.0,0.000000,0.0,0.0
1,B2B,BCE1,D231,702478,PADV-HRCR,2024-10-31,0.0,0.000000,0.0,0.0
2,B2B,BCE1,D231,702478,PADV-HRCR,2024-12-31,0.0,0.000000,0.0,0.0
3,B2B,BCE1,D231,705148,NHR-UTTAM,2024-09-30,0.0,0.000000,0.0,0.0
4,B2B,BCE1,D231,705148,NHR-UTTAM,2024-10-31,0.0,0.000000,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...
2280155,QCOM,QCW2,D463,811279,SAF_CDPRS,2026-05-31,0.0,0.059530,0.0,0.0
2280156,QCOM,QCW2,D463,811279,SAF_CDPRS,2026-06-30,0.0,0.062444,0.0,0.0
2280157,QCOM,QCW2,D463,811287,PA_RSW_SR,2026-04-30,0.0,13.636364,0.0,0.0
2280158,QCOM,QCW2,D463,811287,PA_RSW_SR,2026-05-31,0.0,21.106628,0.0,0.0


In [144]:
vol_cols = [
    'pri_actuals_vol_rum',
    'pri_apo_plan_vol_rum',
    'sec_apo_plan_vol_rum',
    'sec_actuals_vol_rum'
]

actuals_df[vol_cols] = actuals_df[vol_cols].clip(lower=0)

In [145]:
actuals_df.rename(columns = {'final_channel':'channel_name'}, inplace = True)

In [146]:
tmp_df = pd.DataFrame()

for c in ['GT', 'MT', 'ECOM', 'QCOM','B2B']:
    tmp2_df = actuals_df[actuals_df['channel_name'] == c]
    tmp2_df = demand_driver_realign_pskus(tmp2_df, channel=c)
    tmp_df = pd.concat([tmp_df, tmp2_df], ignore_index=True)

In [147]:
tmp_df.duplicated(subset=['channel_name', 'asm_area_code', 'depot_code', 'parent_material_code','month_date']).sum()

23622

In [148]:
tmp_df['month_date'].unique()

<DatetimeArray>
['2023-02-28 00:00:00', '2023-03-31 00:00:00', '2023-04-30 00:00:00',
 '2023-05-31 00:00:00', '2023-06-30 00:00:00', '2023-07-31 00:00:00',
 '2023-08-31 00:00:00', '2023-09-30 00:00:00', '2023-10-31 00:00:00',
 '2023-11-30 00:00:00', '2023-12-31 00:00:00', '2024-01-31 00:00:00',
 '2024-02-29 00:00:00', '2024-03-31 00:00:00', '2024-04-30 00:00:00',
 '2024-05-31 00:00:00', '2024-06-30 00:00:00', '2024-07-31 00:00:00',
 '2024-08-31 00:00:00', '2024-09-30 00:00:00', '2024-10-31 00:00:00',
 '2024-11-30 00:00:00', '2024-12-31 00:00:00', '2025-01-31 00:00:00',
 '2025-02-28 00:00:00', '2025-03-31 00:00:00', '2025-04-30 00:00:00',
 '2025-05-31 00:00:00', '2025-06-30 00:00:00', '2025-07-31 00:00:00',
 '2025-08-31 00:00:00', '2025-09-30 00:00:00', '2025-10-31 00:00:00',
 '2025-11-30 00:00:00', '2026-01-31 00:00:00', '2026-02-28 00:00:00',
 '2023-01-31 00:00:00', '2025-12-31 00:00:00', '2026-03-31 00:00:00',
 '2026-04-30 00:00:00', '2026-05-31 00:00:00', '2026-06-30 00:00:00']
Leng

In [149]:
tmp_df = tmp_df.groupby(
    ['channel_name', 'asm_area_code', 'depot_code', 
     'parent_material_code', 'month_date'], as_index=False
).sum()

In [150]:
tmp_df.duplicated(subset=['channel_name', 'asm_area_code', 'depot_code', 'parent_material_code','month_date']).sum()

0

In [151]:
actuals_df = tmp_df.copy()

del tmp_df, tmp2_df

In [152]:
actuals_df.head()

,channel_name,asm_area_code,depot_code,parent_material_code,month_date,material_group_code,pri_actuals_vol_rum,pri_apo_plan_vol_rum,sec_apo_plan_vol_rum,sec_actuals_vol_rum
0,B2B,BCE1,D231,702478,2024-09-30,PADV-HRCR,0.0,0.0,0.0,0.0
1,B2B,BCE1,D231,702478,2024-10-31,PADV-HRCR,0.0,0.0,0.0,0.0
2,B2B,BCE1,D231,702478,2024-12-31,PADV-HRCR,0.0,0.0,0.0,0.0
3,B2B,BCE1,D231,705148,2024-09-30,NHR-UTTAM,0.0,0.0,0.0,0.0
4,B2B,BCE1,D231,705148,2024-10-31,NHR-UTTAM,0.0,0.0,0.0,0.0


In [153]:
actuals_df['sec_actuals_vol_rum'].sum()

230612137.067

In [154]:
# actuals_df = actuals_df.groupby(['channel_name', 'depot_code', 'parent_material_code',
#        'material_group_code', 'month_date'])[['pri_actuals_vol_rum',
#        'pri_apo_plan_vol_rum', 'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum']].sum().reset_index()
# actuals_df

In [155]:
# actuals_df[actuals_df['channel_name'] == 'GT'].to_csv('trend_GT.csv')

In [156]:
actuals_df = actuals_df.rename(columns={
    'channel_name': 'channel',
    'asm_area_code': 'asm',
    'depot_code': 'depot',
    'parent_material_code': 'psku',
    'sec_apo_plan_vol_rum': 'Consensus Vol',
    'sec_actuals_vol_rum': 'Actuals Vol'
})

In [157]:
actuals_df.columns

Index(['channel', 'asm', 'depot', 'psku', 'month_date', 'material_group_code',
       'pri_actuals_vol_rum', 'pri_apo_plan_vol_rum', 'Consensus Vol',
       'Actuals Vol'],
      dtype='object')

In [158]:
actuals_df = actuals_df.groupby(['channel', 'depot', 'psku', 'month_date'])[['pri_actuals_vol_rum', 'pri_apo_plan_vol_rum', 'Consensus Vol',
       'Actuals Vol']].sum().reset_index()

In [159]:
df = df.groupby(['channel', 'portfolio', 'brand', 'run_month', 'm month',
       'month', 'depot', 'psku'])[['pred vol (roum)', 'prophet vol',
       'rf_vol', 'prophet heuristic vol', 'rf heuristic vol']].sum().reset_index()

In [160]:
df['psku'] = df['psku'].astype(int)
actuals_df['psku'] = actuals_df['psku'].astype(int)

In [161]:
actuals_df.rename(columns = {'month_date':'month'}, inplace = True)

In [162]:
df

,channel,portfolio,brand,run_month,m month,month,depot,psku,pred vol (roum),prophet vol,rf_vol,prophet heuristic vol,rf heuristic vol
0,B2B,0,JH_FRG_L,2026-05-31,M+1,2026-06-30,D117,734910,0.0,0.000000,0.000,0.0,0.0
1,B2B,0,JH_FRG_L,2026-05-31,M+1,2026-06-30,D117,734988,0.0,0.000000,0.000,0.0,0.0
2,B2B,0,JH_FRG_L,2026-05-31,M+1,2026-06-30,D117,734993,0.0,0.000000,0.000,0.0,0.0
3,B2B,0,JH_FRG_L,2026-05-31,M+1,2026-06-30,D117,734995,0.0,0.000000,0.000,0.0,0.0
4,B2B,0,JH_FRG_L,2026-05-31,M+1,2026-06-30,D117,735043,0.0,0.000000,0.000,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
484471,QCOM,Skin Care,PURSNS_ML,2025-11-30,M+4,2026-03-31,D674,810406,0.0,0.000000,0.000,0.0,0.0
484472,QCOM,Skin Care,PURSNS_ML,2025-11-30,M+4,2026-03-31,D674,810407,0.0,0.000000,0.000,0.0,0.0
484473,QCOM,Skin Care,PURSNS_ML,2025-11-30,M+4,2026-03-31,D676,808489,0.0,9.892327,4.895,0.0,0.0
484474,QCOM,Skin Care,PURSNS_ML,2025-11-30,M+4,2026-03-31,D676,809750,0.0,0.000000,0.000,0.0,0.0


In [163]:
# duplicates = actuals_df[actuals_df.duplicated(subset=['channel','depot_code', 'parent_material_code','month_date'], keep=False)]

# # Get indices of duplicates that are NOT PABABY_ML
# indices_to_drop = duplicates[duplicates['material_group_code'] != 'PABABY_ML'].index

# # Remove those rows
# actuals_df = actuals_df.drop(indices_to_drop)

# # Verify no duplicates remain
# print(actuals_df.duplicated(subset=['channel','depot_code', 'parent_material_code','month_date']).sum())

In [164]:
actuals_df[actuals_df.duplicated(subset=['channel', 'depot', 'psku', 'month'], keep=False)]

,channel,depot,psku,month,pri_actuals_vol_rum,pri_apo_plan_vol_rum,Consensus Vol,Actuals Vol


In [165]:
len_before_merge = len(df)
df = df.merge(
    actuals_df.drop([ 'pri_actuals_vol_rum', 'pri_apo_plan_vol_rum'], axis=1),
    on=['channel', 'depot', 'psku', 'month'],
    how='left'
)
assert len_before_merge == len(df)
del len_before_merge

In [166]:
df

,channel,portfolio,brand,run_month,m month,month,depot,psku,pred vol (roum),prophet vol,rf_vol,prophet heuristic vol,rf heuristic vol,Consensus Vol,Actuals Vol
0,B2B,0,JH_FRG_L,2026-05-31,M+1,2026-06-30,D117,734910,0.0,0.000000,0.000,0.0,0.0,NaN,NaN
1,B2B,0,JH_FRG_L,2026-05-31,M+1,2026-06-30,D117,734988,0.0,0.000000,0.000,0.0,0.0,NaN,NaN
2,B2B,0,JH_FRG_L,2026-05-31,M+1,2026-06-30,D117,734993,0.0,0.000000,0.000,0.0,0.0,NaN,NaN
3,B2B,0,JH_FRG_L,2026-05-31,M+1,2026-06-30,D117,734995,0.0,0.000000,0.000,0.0,0.0,NaN,NaN
4,B2B,0,JH_FRG_L,2026-05-31,M+1,2026-06-30,D117,735043,0.0,0.000000,0.000,0.0,0.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
484471,QCOM,Skin Care,PURSNS_ML,2025-11-30,M+4,2026-03-31,D674,810406,0.0,0.000000,0.000,0.0,0.0,NaN,NaN
484472,QCOM,Skin Care,PURSNS_ML,2025-11-30,M+4,2026-03-31,D674,810407,0.0,0.000000,0.000,0.0,0.0,NaN,NaN
484473,QCOM,Skin Care,PURSNS_ML,2025-11-30,M+4,2026-03-31,D676,808489,0.0,9.892327,4.895,0.0,0.0,NaN,NaN
484474,QCOM,Skin Care,PURSNS_ML,2025-11-30,M+4,2026-03-31,D676,809750,0.0,0.000000,0.000,0.0,0.0,NaN,NaN


In [167]:
df[(df['channel'] == 'GT') & (df['month'] == '2026-05-31') & (df['run_month'] == '2026-04-30')]['pred vol (roum)'].sum()

5198880.692278133

In [168]:
df[(df['channel'] == 'GT') & (df['Actuals Vol'].isna())]#['pred vol (roum)'].sum()#.isnull().sum()

,channel,portfolio,brand,run_month,m month,month,depot,psku,pred vol (roum),prophet vol,rf_vol,prophet heuristic vol,rf heuristic vol,Consensus Vol,Actuals Vol
128077,GT,0,PA_SHMP_R,2026-05-31,M+2,2026-07-31,D112,732903,56.100000,0.0,0.0,0.0,0.0,NaN,NaN
128078,GT,0,PA_SHMP_R,2026-05-31,M+2,2026-07-31,D112,732908,29.070000,0.0,0.0,0.0,0.0,NaN,NaN
128079,GT,0,PA_SHMP_R,2026-05-31,M+2,2026-07-31,D112,732925,72.960000,0.0,0.0,0.0,0.0,NaN,NaN
128080,GT,0,PA_SHMP_R,2026-05-31,M+2,2026-07-31,D112,732929,30.600000,0.0,0.0,0.0,0.0,NaN,NaN
128081,GT,0,PA_SHMP_R,2026-05-31,M+2,2026-07-31,D112,732935,29.760000,0.0,0.0,0.0,0.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
287861,GT,Skin Care,PAD_BDYOL,2026-05-31,M+4,2026-09-30,D234,719116,0.000000,0.0,0.0,0.0,0.0,NaN,NaN
287862,GT,Skin Care,PAD_BDYOL,2026-05-31,M+4,2026-09-30,D236,719117,0.182498,0.0,0.0,0.0,0.0,NaN,NaN
287863,GT,Skin Care,PAD_BDYOL,2026-05-31,M+4,2026-09-30,D236,719118,0.093333,0.0,0.0,0.0,0.0,NaN,NaN
287864,GT,Skin Care,PAD_BDYOL,2026-05-31,M+4,2026-09-30,D463,719116,0.272000,0.0,0.0,0.0,0.0,NaN,NaN


In [169]:
# actuals_df[(actuals_df['asm'] == 'KARN') & (actuals_df['psku'] == 718314)]
actuals_df['key'] =  actuals_df['depot'] + actuals_df['psku'].astype(str)
df['key'] = df['depot'] + df['psku'].astype(str)


In [170]:
def read_qtr_ind_rate_table():
    query = """select * from DWH_SAP_INDEX_TURNOVER_MONTHWISE 
                where latest_rate_flag=1 and company_code='MIL'"""
    qtr_ind_rate_data = pd.read_sql(con=prod_conn, sql=query)
    qtr_ind_rate_data.columns = qtr_ind_rate_data.columns.str.lower()
    qtr_ind_rate =  qtr_ind_rate_data[['date', 'brand_code', 'turnover']]
    qtr_ind_rate = qtr_ind_rate.rename(columns= {'date':'month_date', 'turnover':'qtr_ind_rate'})
    
    return qtr_ind_rate

qtr_ind_rate_df = read_qtr_ind_rate_table()
qtr_ind_rate_df.head()

,month_date,brand_code,qtr_ind_rate
0,2027-03-31,PA_CN_HGO,488.152
1,2027-03-31,TRU_RAWDF,800.000
2,2027-03-31,TRU_PDRFR,850.570
3,2027-03-31,TRU_OATS,177.070
4,2027-03-31,TRU_QUINO,204.750


In [171]:
qtr_ind_rate_df[qtr_ind_rate_df['brand_code'] == 'PCNO(R)']

,month_date,brand_code,qtr_ind_rate
70,2027-03-31,PCNO(R),349274.00142


In [172]:
len_before_merge = len(df)
df = df.merge(
    qtr_ind_rate_df.drop('month_date', axis=1).rename(
        columns={'brand_code': 'brand', 'qtr_ind_rate': 'Index Rate'}
    ),
    on=['brand'], 
    how='left'
)
assert len_before_merge == len(df)
del len_before_merge

In [173]:
df['brand'].unique()

array(['JH_FRG_L', 'PA_SHMP_R', 'PA_SHMP_S', 'NHR-UTTAM', 'PCNO FLEX',
       'PCNO(R)', 'CO_SO_VCN', 'SAF-MUSLI', 'SAFF OATS', 'SAFF SALT',
       'SAFF_ODLS', 'SAF_HONEY', 'SAF_MAYO', 'SAF_MILET', 'SAF_PNBTR',
       'SFOAT-CUP', 'SFOATS-FL', 'SFOATS_MG', 'SF_IM_CHY', 'SF_MNCHPS',
       'SF_SOYACN', 'ADV-AHO-R', 'H&C', 'H&C_ALMND', 'NHR NSJ H',
       'NHR-SABDM', 'NHR_ALOAM', 'NHR_SSAHO', 'NIHAR NHO', 'PA-ALO-HO',
       'PADV-HOT', 'PADVJAS-R', 'PADV_AMRO', 'PADV_SMPN', 'PA_AMVITE',
       'PA_CN_HO', 'PA_EXT_ML', 'PA_JASGLD', 'P_AL_GOLD', 'P_EN_ALM',
       'P_EN_BGHB', 'P_EN_CRSH', 'P_EN_RSMR', 'JH_SCR_KG', 'JH_SCR_L',
       'BRD_BDOIL', 'BRD_BDSPR', 'BRD_DOGAS', 'BRD_FSWSH', 'BRD_HROIL',
       'BRD_HRWAX', 'BRD_PERFM', 'PADV-HRCR', 'SW HRGEL', 'SW HSPRY',
       'SW NOGAS', 'SW STLDEO', 'SW_HR_WAX', 'NHR_VTEHO', 'LVN_SHMP',
       'MALO-NATU', 'MALT-NATU', 'REV.LQDST', 'REV.ST.', 'REV_LQFRG',
       'HC SNS', 'LIVON', 'LIVON S-R', 'LVN_SR_DR', 'SAFF ACTV',
       'SAFF GOLD',

In [174]:
df

,channel,portfolio,brand,run_month,m month,month,depot,psku,pred vol (roum),prophet vol,rf_vol,prophet heuristic vol,rf heuristic vol,Consensus Vol,Actuals Vol,key,Index Rate
0,B2B,0,JH_FRG_L,2026-05-31,M+1,2026-06-30,D117,734910,0.0,0.000000,0.000,0.0,0.0,NaN,NaN,D117734910,NaN
1,B2B,0,JH_FRG_L,2026-05-31,M+1,2026-06-30,D117,734988,0.0,0.000000,0.000,0.0,0.0,NaN,NaN,D117734988,NaN
2,B2B,0,JH_FRG_L,2026-05-31,M+1,2026-06-30,D117,734993,0.0,0.000000,0.000,0.0,0.0,NaN,NaN,D117734993,NaN
3,B2B,0,JH_FRG_L,2026-05-31,M+1,2026-06-30,D117,734995,0.0,0.000000,0.000,0.0,0.0,NaN,NaN,D117734995,NaN
4,B2B,0,JH_FRG_L,2026-05-31,M+1,2026-06-30,D117,735043,0.0,0.000000,0.000,0.0,0.0,NaN,NaN,D117735043,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
484471,QCOM,Skin Care,PURSNS_ML,2025-11-30,M+4,2026-03-31,D674,810406,0.0,0.000000,0.000,0.0,0.0,NaN,NaN,D674810406,1779.273746
484472,QCOM,Skin Care,PURSNS_ML,2025-11-30,M+4,2026-03-31,D674,810407,0.0,0.000000,0.000,0.0,0.0,NaN,NaN,D674810407,1779.273746
484473,QCOM,Skin Care,PURSNS_ML,2025-11-30,M+4,2026-03-31,D676,808489,0.0,9.892327,4.895,0.0,0.0,NaN,NaN,D676808489,1779.273746
484474,QCOM,Skin Care,PURSNS_ML,2025-11-30,M+4,2026-03-31,D676,809750,0.0,0.000000,0.000,0.0,0.0,NaN,NaN,D676809750,1779.273746


In [175]:
df.isna().sum()

channel                       0
portfolio                     0
brand                         0
run_month                     0
m month                       0
month                         0
depot                         0
psku                          0
pred vol (roum)               0
prophet vol                   0
rf_vol                        0
prophet heuristic vol         0
rf heuristic vol              0
Consensus Vol            152782
Actuals Vol              152782
key                           0
Index Rate                   76
dtype: int64

In [176]:
df['Consensus Vol'] = df['Consensus Vol'].fillna(0)
df['Actuals Vol'] = df['Actuals Vol'].fillna(0)

In [177]:
df.columns

Index(['channel', 'portfolio', 'brand', 'run_month', 'm month', 'month',
       'depot', 'psku', 'pred vol (roum)', 'prophet vol', 'rf_vol',
       'prophet heuristic vol', 'rf heuristic vol', 'Consensus Vol',
       'Actuals Vol', 'key', 'Index Rate'],
      dtype='object')

In [178]:
df.rename(columns={'pred vol (roum)': 'Stat Vol'}, inplace=True)

In [179]:
df['Stat Val'] = df['Stat Vol'] * df['Index Rate'] / (10 ** 7)
df['Consensus Val'] = df['Consensus Vol'] * df['Index Rate'] / (10 ** 7)
df['Actuals Val'] = df['Actuals Vol'] * df['Index Rate'] / (10 ** 7)

In [180]:
# df[(df['channel'] == 'GT') & (df['Actuals Vol'].isna()) & (df['run_month'] == '2026-03-31') & (df['m month'] == 'M+1')]['Stat Val'].sum()

In [181]:
df[(df['run_month'] == '2026-04-30') & (df['m month'] == 'M+1') & (df['channel'] == 'GT')]['Actuals Val'].sum()

456.02810850581477

In [182]:
df['Stat Error'] = df['Stat Val'] - df['Actuals Val']
df['Consensus Error'] = df['Consensus Val'] - df['Actuals Val']

df['Stat Abs Error'] = np.abs(df['Stat Error'])
df['Consensus Abs Error'] = np.abs(df['Consensus Error'])

In [183]:
df.columns

Index(['channel', 'portfolio', 'brand', 'run_month', 'm month', 'month',
       'depot', 'psku', 'Stat Vol', 'prophet vol', 'rf_vol',
       'prophet heuristic vol', 'rf heuristic vol', 'Consensus Vol',
       'Actuals Vol', 'key', 'Index Rate', 'Stat Val', 'Consensus Val',
       'Actuals Val', 'Stat Error', 'Consensus Error', 'Stat Abs Error',
       'Consensus Abs Error'],
      dtype='object')

In [184]:
for col in ['prophet vol', 'rf_vol',
       'prophet heuristic vol', 'rf heuristic vol']:
    df[f'{col}_value'] = df[col] * df['Index Rate'] / (10 ** 7)

In [185]:
df['stat_bias'] = df['Stat Error']/df['Actuals Val']

In [186]:
df = df.fillna(0)

In [187]:
df['run_month'].unique()

<DatetimeArray>
['2026-05-31 00:00:00', '2025-10-31 00:00:00', '2025-11-30 00:00:00',
 '2025-12-31 00:00:00', '2026-01-31 00:00:00', '2026-02-28 00:00:00',
 '2026-03-31 00:00:00', '2026-04-30 00:00:00']
Length: 8, dtype: datetime64[ns]

In [188]:
df = df[df['m month'] == 'M+1']#.isnull().sum()

In [189]:
import numpy as np
import pandas as pd

df['stat_bias'] = (
    df['stat_bias']
    .replace([np.inf, -np.inf], 0)
    .fillna(0)
)

bins = [-np.inf, -0.15, -0.10, -0.05, 0, 0.05, 0.10, 0.15, np.inf]
labels = [
    '< -15%',
    '-15% to -10%',
    '-10% to -5%',
    '-5% to 0%',
    '0% to 5%',
    '5% to 10%',
    '10% to 15%',
    '> 15%'
]

df['stat_bias_bucket'] = pd.cut(
    df['stat_bias'],
    bins=bins,
    labels=labels,
    right=False   # 👈 key change
)


In [190]:
df

,channel,portfolio,brand,run_month,m month,month,depot,psku,Stat Vol,prophet vol,...,Stat Error,Consensus Error,Stat Abs Error,Consensus Abs Error,prophet vol_value,rf_vol_value,prophet heuristic vol_value,rf heuristic vol_value,stat_bias,stat_bias_bucket
0,B2B,0,JH_FRG_L,2026-05-31,M+1,2026-06-30,D117,734910,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0% to 5%
1,B2B,0,JH_FRG_L,2026-05-31,M+1,2026-06-30,D117,734988,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0% to 5%
2,B2B,0,JH_FRG_L,2026-05-31,M+1,2026-06-30,D117,734993,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0% to 5%
3,B2B,0,JH_FRG_L,2026-05-31,M+1,2026-06-30,D117,734995,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0% to 5%
4,B2B,0,JH_FRG_L,2026-05-31,M+1,2026-06-30,D117,735043,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0% to 5%
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
484057,QCOM,Skin Care,PURSNS_ML,2025-11-30,M+1,2025-12-31,D674,810406,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0% to 5%
484058,QCOM,Skin Care,PURSNS_ML,2025-11-30,M+1,2025-12-31,D674,810407,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0% to 5%
484059,QCOM,Skin Care,PURSNS_ML,2025-11-30,M+1,2025-12-31,D676,808489,0.0,23.308381,...,0.0,0.0,0.0,0.0,0.004147,0.001373,0.0,0.0,0.0,0% to 5%
484060,QCOM,Skin Care,PURSNS_ML,2025-11-30,M+1,2025-12-31,D676,809750,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0% to 5%


In [191]:
df['forecast_granularity'] = 'Depot x PSKU'
df['forecast_type'] = 'secondary'

In [192]:
cols = ['forecast_granularity', 'forecast_type'] + [
    c for c in df.columns
    if c not in ['forecast_granularity', 'forecast_type']
]

df = df[cols]

In [193]:
df['run_month'].unique()

<DatetimeArray>
['2026-05-31 00:00:00', '2025-10-31 00:00:00', '2025-11-30 00:00:00',
 '2025-12-31 00:00:00', '2026-01-31 00:00:00', '2026-02-28 00:00:00',
 '2026-03-31 00:00:00', '2026-04-30 00:00:00']
Length: 8, dtype: datetime64[ns]

### Qcom

### Helper Functions

In [194]:
realignment_df = pd.read_sql(
    """select * from trn_mil_asm_psku_realignment""",
    dev_conn
)
realignment_df.columns = realignment_df.columns.str.lower()

# realignment_df = realignment_df[
#     realignment_df['channel'].isin(['QCOM', 'QCOM B2C', 'ALL'])]


def realign_pskus(data, column, channel='QCOM'):
    realignment_data = realignment_df.copy()
    realignment_data = realignment_data[
        realignment_data['channel'].isin([channel, channel + ' B2C', 'ALL'])
    ]

    assert realignment_data['psku old'].dtype == 'int64'
    assert realignment_data['psku new'].dtype == 'int64'
    realignment_data = realignment_data[['psku old', 'psku new']].drop_duplicates()
    realignment_data = realignment_data.set_index('psku old').to_dict()['psku new']

    
    data[column] = data[column].astype(int)

    for old_psku, new_psku in realignment_data.items():
        data.loc[
            data[column] == old_psku, column
        ] = new_psku

    return data

### Forecast

In [195]:
qcom_df = pd.read_sql(
    """select * from TRN_MIL_DF_OFT2PRIM_SHARED""",
    dev_conn
)
qcom_df.columns = qcom_df.columns.str.lower()
qcom_df['run_month'] = pd.to_datetime(qcom_df['run_month'])
qcom_df['month'] = pd.to_datetime(qcom_df['month'])

In [196]:
qcom_df#[qcom_df['Channel']=='Ecom']

,month,depot,psku,brand,portfolio,m month,run_month,calculated primary vol,channel
0,2026-04-30,D111,709567,SAFF OATS,Foods,M+1,2026-03-31,0.000000,Ecom
1,2026-04-30,D111,718288,SAFF GOLD,Saffola Oils,M+1,2026-03-31,0.000000,Ecom
2,2026-04-30,D111,718297,PCNO(R),CNO,M+1,2026-03-31,0.000000,Ecom
3,2026-04-30,D111,718299,PCNO(R),CNO,M+1,2026-03-31,0.000000,Ecom
4,2026-04-30,D111,718303,REV.LQDST,Others,M+1,2026-03-31,0.000000,Ecom
...,...,...,...,...,...,...,...,...,...
578929,2026-07-31,D677,810738,PABABY_GM,Skin Care,M+4,2026-03-31,13.968941,Qcom
578930,2026-07-31,D677,810805,PABABY_GM,Skin Care,M+4,2026-03-31,0.000000,Qcom
578931,2026-07-31,D677,810971,PA_ESS_HO,Hair Oils,M+4,2026-03-31,0.000000,Qcom
578932,2026-07-31,D677,811169,SW_SGPRF,Male Grooming,M+4,2026-03-31,0.000000,Qcom


### Actuals

In [197]:
realignment_df = pd.read_sql(
    """select * from trn_mil_asm_psku_realignment""",
    dev_conn
)
realignment_df.columns = realignment_df.columns.str.lower()

# realignment_df = realignment_df[
#     realignment_df['channel'].isin(['QCOM', 'QCOM B2C', 'ALL'])]


def realign_pskus(data, column, channel='QCOM'):
    realignment_data = realignment_df.copy()
    realignment_data = realignment_data[
        realignment_data['channel'].isin([channel, channel + ' B2C', 'ALL'])
    ]

    assert realignment_data['psku old'].dtype == 'int64'
    assert realignment_data['psku new'].dtype == 'int64'
    realignment_data = realignment_data[['psku old', 'psku new']].drop_duplicates()
    realignment_data = realignment_data.set_index('psku old').to_dict()['psku new']

    
    data[column] = data[column].astype(int)

    for old_psku, new_psku in realignment_data.items():
        data.loc[
            data[column] == old_psku, column
        ] = new_psku

    return data

In [198]:
depot_psku_primary_query = """
SELECT
    CM.depot_code,
    MM.parent_material_code,
    MM.material_group_code,
    LAST_DAY(MESR.month_date) AS month_date,
    SUM(pri_actuals_vol_rum) AS pri_actuals_vol_rum,
    SUM(pri_apo_plan_vol_rum) AS pri_apo_plan_vol_rum,
    SUM(MESR.sec_apo_plan_vol_rum) AS sec_apo_plan_vol_rum,
    SUM(MESR.sec_actuals_vol_rum) AS sec_actuals_vol_rum
FROM
    dwh_bpm_dist_sku_daily MESR
JOIN 
(
    SELECT
        customer,
        chain_type,
        chain
    FROM
        mst_chain_master
    WHERE
        chain_type = 'E Com B2C' AND 
        chain IN ('Grofers', 'Zepto', 'Kiranakart Technologies', 'Swiggy', 'ZEPTO')
) MCM ON MESR.distributor_code = MCM.customer
JOIN
(
    SELECT
        material_code,
        parent_material_code,
        material_group_code,
        uom_reporting,
        vol_per_unit
    FROM 
        mst_material
    WHERE
        company_code='MIL' AND
        latest_record_ind=1
) MM ON MESR.material_code = MM.material_code
JOIN
(
    SELECT DISTINCT
        customer_code,
        depot_code
    FROM
        mst_customer
    WHERE
        company_code='MIL' AND
        latest_record_ind=1
) CM on MESR.distributor_code = CM.customer_code
where month_date BETWEEN '2023-01-01' AND '2026-06-30' 
GROUP BY 1, 2, 3, 4
ORDER BY 1, 3, 2, 4
"""

depot_psku_primary_df = pd.read_sql(
    depot_psku_primary_query,
    prod_conn
)
depot_psku_primary_df.columns = depot_psku_primary_df.columns.str.lower()
depot_psku_primary_df['parent_material_code'] = depot_psku_primary_df['parent_material_code'].astype(int)
depot_psku_primary_df['month_date'] = pd.to_datetime(depot_psku_primary_df['month_date'])
depot_psku_primary_df = realign_pskus(depot_psku_primary_df.copy(), 'parent_material_code')
depot_psku_primary_df = depot_psku_primary_df.groupby(
    ['depot_code', 'parent_material_code', 'material_group_code', 'month_date'], as_index=False, dropna=False
).sum()
depot_psku_primary_df = depot_psku_primary_df[depot_psku_primary_df['parent_material_code'] != 715096]
depot_psku_primary_df.duplicated(subset=['depot_code', 'parent_material_code', 'month_date']).sum()
for col in ['pri_actuals_vol_rum', 'pri_apo_plan_vol_rum', 'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum']:
    if depot_psku_primary_df[col].min() < 0:
        print(col)

for col in ['pri_actuals_vol_rum', 'pri_apo_plan_vol_rum', 'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum']:
    depot_psku_primary_df[col] = depot_psku_primary_df[col].clip(lower=0)

pri_actuals_vol_rum
sec_actuals_vol_rum


In [199]:
depot_psku_primary_df['channel'] = 'QCOM'
depot_psku_primary_df_qcom = depot_psku_primary_df.copy()

In [200]:
depot_psku_primary_df_qcom.to_csv('Trend_qcom.csv')

In [201]:
depot_psku_primary_query = """
SELECT
    CM.depot_code,
    MM.parent_material_code,
    MM.material_group_code,
    LAST_DAY(MESR.month_date) AS month_date,
    SUM(pri_actuals_vol_rum) AS pri_actuals_vol_rum,
    SUM(pri_apo_plan_vol_rum) AS pri_apo_plan_vol_rum,
    SUM(MESR.sec_apo_plan_vol_rum) AS sec_apo_plan_vol_rum,
    SUM(MESR.sec_actuals_vol_rum) AS sec_actuals_vol_rum
FROM
    dwh_bpm_dist_sku_daily MESR
JOIN 
(
    SELECT
        customer,
        chain_type,
        chain
    FROM
        mst_chain_master
    WHERE
        chain_type = 'E Com B2C' AND 
        chain IN ('Flipkart-National', 'Flipkart-Grocery', 'Big basket B2C', 'RK WORLDINFOCOM', 'Amazon B2C', 
        'FATEHPURIA HYGIENE', 'Nykaa', 'Flipkart-Minutes', 'Purplle', 'Myntra', 'Dealshare', 'FlipkartGrocery', 'City Mall', '1MG', 
        'ARIPL', 'First Cry', 'Meesho', 'RKWorld', 'CITIMALL', 'Firstcry', 'EMAZING DEALS', 'MYNTRA')
) MCM ON MESR.distributor_code = MCM.customer
JOIN
(
    SELECT
        material_code,
        parent_material_code,
        material_group_code,
        uom_reporting,
        vol_per_unit
    FROM 
        mst_material
    WHERE
        company_code='MIL' AND
        latest_record_ind=1
) MM ON MESR.material_code = MM.material_code
JOIN
(
    SELECT DISTINCT
        customer_code,
        depot_code
    FROM
        mst_customer
    WHERE
        company_code='MIL' AND
        latest_record_ind=1
) CM on MESR.distributor_code = CM.customer_code
where month_date BETWEEN '2025-12-01' AND '2026-06-30' 
GROUP BY 1, 2, 3, 4
ORDER BY 1, 3, 2, 4
"""

depot_psku_primary_df = pd.read_sql(
    depot_psku_primary_query,
    prod_conn
)
depot_psku_primary_df.columns = depot_psku_primary_df.columns.str.lower()
depot_psku_primary_df['parent_material_code'] = depot_psku_primary_df['parent_material_code'].astype(int)
depot_psku_primary_df['month_date'] = pd.to_datetime(depot_psku_primary_df['month_date'])
depot_psku_primary_df = realign_pskus(depot_psku_primary_df.copy(), 'parent_material_code', channel='ECOM')
depot_psku_primary_df = depot_psku_primary_df.groupby(
    ['depot_code', 'parent_material_code', 'material_group_code', 'month_date'], as_index=False, dropna=False
).sum()
depot_psku_primary_df = depot_psku_primary_df[depot_psku_primary_df['parent_material_code'] != 715096]
depot_psku_primary_df.duplicated(subset=['depot_code', 'parent_material_code', 'month_date']).sum()
for col in ['pri_actuals_vol_rum', 'pri_apo_plan_vol_rum', 'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum']:
    if depot_psku_primary_df[col].min() < 0:
        print(col)

for col in ['pri_actuals_vol_rum', 'pri_apo_plan_vol_rum', 'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum']:
    depot_psku_primary_df[col] = depot_psku_primary_df[col].clip(lower=0)

pri_actuals_vol_rum
sec_actuals_vol_rum


In [202]:
depot_psku_primary_df['channel'] = 'ECOM'
depot_psku_primary_df_ecom = depot_psku_primary_df.copy()

In [203]:
depot_psku_primary_df = pd.concat([depot_psku_primary_df_qcom, depot_psku_primary_df_ecom], ignore_index=True)
depot_psku_primary_df

,depot_code,parent_material_code,material_group_code,month_date,pri_actuals_vol_rum,pri_apo_plan_vol_rum,sec_apo_plan_vol_rum,sec_actuals_vol_rum,channel
0,D112,715098,CO_SO_PCP,2023-07-31,0.0,0.000000,1.762,0.0,QCOM
1,D112,715098,CO_SO_PCP,2023-08-31,0.0,0.000000,1.762,0.0,QCOM
2,D112,715099,CO_SO_PCP,2023-07-31,0.0,0.000000,1.762,0.0,QCOM
3,D112,715099,CO_SO_PCP,2023-08-31,0.0,0.000000,1.762,0.0,QCOM
4,D112,715100,CO_SO_PCP,2023-02-28,0.0,0.000000,0.284,0.0,QCOM
...,...,...,...,...,...,...,...,...,...
180764,D677,811279,SAF_CDPRS,2026-06-30,0.0,0.042710,0.000,0.0,ECOM
180765,D677,811287,PA_RSW_SR,2026-04-30,0.0,11.800708,0.000,0.0,ECOM
180766,D677,811287,PA_RSW_SR,2026-05-31,0.0,10.794646,0.000,0.0,ECOM
180767,D677,811287,PA_RSW_SR,2026-06-30,0.0,12.332112,0.000,0.0,ECOM


In [204]:
actuals_df = depot_psku_primary_df.copy()

In [205]:
actuals_df.duplicated(subset=['channel','depot_code', 'parent_material_code','month_date']).sum()

440

In [206]:
# For duplicates, keep only the row with PABABY_ML material_group_code
duplicates = actuals_df[actuals_df.duplicated(subset=['channel','depot_code', 'parent_material_code','month_date'], keep=False)]

# Get indices of duplicates that are NOT PABABY_ML
indices_to_drop = duplicates[duplicates['material_group_code'] != 'PABABY_ML'].index

# Remove those rows
actuals_df = actuals_df.drop(indices_to_drop)

# Verify no duplicates remain
print(actuals_df.duplicated(subset=['channel','depot_code', 'parent_material_code','month_date']).sum())

0


In [207]:
actuals_df = actuals_df.groupby(['channel','depot_code', 'parent_material_code',
       'material_group_code', 'month_date'])[['pri_actuals_vol_rum',
       'pri_apo_plan_vol_rum', 'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum']].sum().reset_index()
actuals_df

,channel,depot_code,parent_material_code,material_group_code,month_date,pri_actuals_vol_rum,pri_apo_plan_vol_rum,sec_apo_plan_vol_rum,sec_actuals_vol_rum
0,ECOM,D112,718287,PCNO(R),2025-12-31,0.000,0.023463,0.0000,0.000
1,ECOM,D112,718288,SAFF GOLD,2026-03-31,0.000,0.000000,1.9086,0.000
2,ECOM,D112,718288,SAFF GOLD,2026-05-31,0.000,0.028975,0.0000,0.000
3,ECOM,D112,718299,PCNO(R),2026-04-30,0.000,0.038390,0.0618,0.000
4,ECOM,D112,718299,PCNO(R),2026-05-31,0.057,0.074845,0.0557,0.057
...,...,...,...,...,...,...,...,...,...
180015,QCOM,D677,811279,SAF_CDPRS,2026-05-31,0.000,0.041760,0.0092,0.000
180016,QCOM,D677,811279,SAF_CDPRS,2026-06-30,0.000,0.045348,0.0000,0.000
180017,QCOM,D677,811287,PA_RSW_SR,2026-04-30,0.000,8.823530,0.0000,0.000
180018,QCOM,D677,811287,PA_RSW_SR,2026-05-31,0.000,14.774640,0.0000,0.000


In [208]:
actuals_df.columns

Index(['channel', 'depot_code', 'parent_material_code', 'material_group_code',
       'month_date', 'pri_actuals_vol_rum', 'pri_apo_plan_vol_rum',
       'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum'],
      dtype='object')

In [209]:
actuals_df = actuals_df.rename(columns={
    'depot_code': 'depot',
    'parent_material_code': 'psku',
    'sec_apo_plan_vol_rum': 'Consensus Vol',
    'sec_actuals_vol_rum': 'Actuals Vol',
    'month_date': 'month'
})

In [210]:
actuals_df.head()

,channel,depot,psku,material_group_code,month,pri_actuals_vol_rum,pri_apo_plan_vol_rum,Consensus Vol,Actuals Vol
0,ECOM,D112,718287,PCNO(R),2025-12-31,0.000,0.023463,0.0000,0.000
1,ECOM,D112,718288,SAFF GOLD,2026-03-31,0.000,0.000000,1.9086,0.000
2,ECOM,D112,718288,SAFF GOLD,2026-05-31,0.000,0.028975,0.0000,0.000
3,ECOM,D112,718299,PCNO(R),2026-04-30,0.000,0.038390,0.0618,0.000
4,ECOM,D112,718299,PCNO(R),2026-05-31,0.057,0.074845,0.0557,0.057


In [211]:
#qcom_df.rename(columns = {'Channel':'channel'}, inplace = True)

In [212]:
qcom_df['psku'] = qcom_df['psku'].astype(int)
actuals_df['psku'] = actuals_df['psku'].astype(int)

In [213]:
# actuals_df = actuals_df[actuals_df['month_date'] == '2025-12-31']
# actuals_df

In [214]:
qcom_df['run_month'].unique()

<DatetimeArray>
['2026-03-31 00:00:00', '2026-04-30 00:00:00', '2026-05-31 00:00:00',
 '2025-11-30 00:00:00', '2025-12-31 00:00:00', '2026-01-31 00:00:00',
 '2026-02-28 00:00:00']
Length: 7, dtype: datetime64[ns]

In [215]:
qcom_df['channel'] = qcom_df['channel'].str.upper()
qcom_df

,month,depot,psku,brand,portfolio,m month,run_month,calculated primary vol,channel
0,2026-04-30,D111,709567,SAFF OATS,Foods,M+1,2026-03-31,0.000000,ECOM
1,2026-04-30,D111,718288,SAFF GOLD,Saffola Oils,M+1,2026-03-31,0.000000,ECOM
2,2026-04-30,D111,718297,PCNO(R),CNO,M+1,2026-03-31,0.000000,ECOM
3,2026-04-30,D111,718299,PCNO(R),CNO,M+1,2026-03-31,0.000000,ECOM
4,2026-04-30,D111,718303,REV.LQDST,Others,M+1,2026-03-31,0.000000,ECOM
...,...,...,...,...,...,...,...,...,...
578929,2026-07-31,D677,810738,PABABY_GM,Skin Care,M+4,2026-03-31,13.968941,QCOM
578930,2026-07-31,D677,810805,PABABY_GM,Skin Care,M+4,2026-03-31,0.000000,QCOM
578931,2026-07-31,D677,810971,PA_ESS_HO,Hair Oils,M+4,2026-03-31,0.000000,QCOM
578932,2026-07-31,D677,811169,SW_SGPRF,Male Grooming,M+4,2026-03-31,0.000000,QCOM


In [216]:
len_before_merge = len(qcom_df)
qcom_df = qcom_df.merge(
    actuals_df.drop([ 'material_group_code', 'pri_actuals_vol_rum', 'pri_apo_plan_vol_rum'], axis=1),
    on=['channel','depot', 'psku', 'month'],
    how='left'
)
assert len_before_merge == len(qcom_df)
del len_before_merge

In [217]:
def read_qtr_ind_rate_table():
    query = """select * from DWH_SAP_INDEX_TURNOVER_MONTHWISE 
                where latest_rate_flag=1 and company_code='MIL'"""
    qtr_ind_rate_data = pd.read_sql(con=prod_conn, sql=query)
    qtr_ind_rate_data.columns = qtr_ind_rate_data.columns.str.lower()
    qtr_ind_rate =  qtr_ind_rate_data[['date', 'brand_code', 'turnover']]
    qtr_ind_rate = qtr_ind_rate.rename(columns= {'date':'month_date', 'turnover':'qtr_ind_rate'})
    
    return qtr_ind_rate

qtr_ind_rate_df = read_qtr_ind_rate_table()
qtr_ind_rate_df.head()

,month_date,brand_code,qtr_ind_rate
0,2027-03-31,PA_CN_HGO,488.152
1,2027-03-31,TRU_RAWDF,800.000
2,2027-03-31,TRU_PDRFR,850.570
3,2027-03-31,TRU_OATS,177.070
4,2027-03-31,TRU_QUINO,204.750


In [218]:
len_before_merge = len(qcom_df)
qcom_df = qcom_df.merge(
    qtr_ind_rate_df.drop('month_date', axis=1).rename(
        columns={'brand_code': 'brand', 'qtr_ind_rate': 'Index Rate'}
    ),
    on=['brand'], 
    how='left'
)
assert len_before_merge == len(qcom_df)
del len_before_merge

In [219]:
qcom_df.isna().sum()

month                          0
depot                          0
psku                           0
brand                          0
portfolio                      0
m month                        0
run_month                      0
calculated primary vol         0
channel                        0
Consensus Vol             441739
Actuals Vol               441739
Index Rate                     0
dtype: int64

In [220]:
qcom_df['Consensus Vol'] = qcom_df['Consensus Vol'].fillna(0)
qcom_df['Actuals Vol'] = qcom_df['Actuals Vol'].fillna(0)

In [221]:
qcom_df['Consensus Vol'].max()

21077.84

In [222]:
qcom_df.rename(columns={'calculated primary vol': 'Stat Vol'}, inplace=True)
qcom_df

,month,depot,psku,brand,portfolio,m month,run_month,Stat Vol,channel,Consensus Vol,Actuals Vol,Index Rate
0,2026-04-30,D111,709567,SAFF OATS,Foods,M+1,2026-03-31,0.000000,ECOM,0.0,0.0,127515.619406
1,2026-04-30,D111,718288,SAFF GOLD,Saffola Oils,M+1,2026-03-31,0.000000,ECOM,0.0,0.0,138865.260689
2,2026-04-30,D111,718297,PCNO(R),CNO,M+1,2026-03-31,0.000000,ECOM,0.0,0.0,349274.001420
3,2026-04-30,D111,718299,PCNO(R),CNO,M+1,2026-03-31,0.000000,ECOM,0.0,0.0,349274.001420
4,2026-04-30,D111,718303,REV.LQDST,Others,M+1,2026-03-31,0.000000,ECOM,0.0,0.0,247926.608903
...,...,...,...,...,...,...,...,...,...,...,...,...
578929,2026-07-31,D677,810738,PABABY_GM,Skin Care,M+4,2026-03-31,13.968941,QCOM,0.0,0.0,366.484998
578930,2026-07-31,D677,810805,PABABY_GM,Skin Care,M+4,2026-03-31,0.000000,QCOM,0.0,0.0,366.484998
578931,2026-07-31,D677,810971,PA_ESS_HO,Hair Oils,M+4,2026-03-31,0.000000,QCOM,0.0,0.0,12860.631072
578932,2026-07-31,D677,811169,SW_SGPRF,Male Grooming,M+4,2026-03-31,0.000000,QCOM,0.0,0.0,1712.605337


In [223]:
qcom_df['Stat Val'] = qcom_df['Stat Vol'] * qcom_df['Index Rate'] / (10 ** 7)
qcom_df['Consensus Val'] = qcom_df['Consensus Vol'] * qcom_df['Index Rate'] / (10 ** 7)
qcom_df['Actuals Val'] = qcom_df['Actuals Vol'] * qcom_df['Index Rate'] / (10 ** 7)

In [224]:
qcom_df['channel'].unique()

array(['ECOM', 'QCOM'], dtype=object)

In [227]:
qcom_df[(qcom_df['month'] == '2026-06-30') & (qcom_df['channel']=='QCOM') & (qcom_df['run_month']=='2026-05-31')]['Actuals Val'].sum()

32.844680148042876

In [228]:
qcom_df['Stat Error'] = qcom_df['Stat Val'] - qcom_df['Actuals Val']
qcom_df['Consensus Error'] = qcom_df['Consensus Val'] - qcom_df['Actuals Val']

qcom_df['Stat Abs Error'] = np.abs(qcom_df['Stat Error'])
qcom_df['Consensus Abs Error'] = np.abs(qcom_df['Consensus Error'])

In [229]:
qcom_df['stat_bias'] = qcom_df['Stat Error']/qcom_df['Actuals Val']
qcom_df = qcom_df.fillna(0)

import numpy as np
import pandas as pd

qcom_df['stat_bias'] = (
    qcom_df['stat_bias']
    .replace([np.inf, -np.inf], 0)
    .fillna(0)
)

bins = [-np.inf, -0.15, -0.10, -0.05, 0, 0.05, 0.10, 0.15, np.inf]
labels = [
    '< -15%',
    '-15% to -10%',
    '-10% to -5%',
    '-5% to 0%',
    '0% to 5%',
    '5% to 10%',
    '10% to 15%',
    '> 15%'
]

qcom_df['stat_bias_bucket'] = pd.cut(
    qcom_df['stat_bias'],
    bins=bins,
    labels=labels,
    right=False  
)


In [109]:
# delivery_df = pd.read_csv('/data/aman_singh/acuuracy_check/Export View of Month Vol & APO.csv')

# delivery_df['Channel'] = delivery_df['Channel'].replace({
#     'E-Commerce': 'ECOM', 'Q-Commerce': 'QCOM'})
# delivery_df
# delivery_df['delivery_vol'] = delivery_df['01-12-2025 Del Vol'].map(lambda x:0 if x.strip() == '-' else float(x.strip().replace(',','')))
# delivery_df = delivery_df[delivery_df['Channel'] == 'QCOM']
# delivery_df = delivery_df.groupby(['Depot','PSKU'])['delivery_vol'].sum().reset_index(
# )#.rename(columns = {'01-12-2025 Del Vol':'delivery_vol'})

# len_before_merge = len(qcom_df)
# qcom_df = qcom_df.merge(
#     delivery_df,
#     on=['Depot', 'PSKU'],
#     how='left'
# )
# assert len_before_merge == len(qcom_df)
# del len_before_merge

# qcom_df['Delivery Val'] = qcom_df['delivery_vol'] * qcom_df['Index Rate'] / (10 ** 7)

# qcom_df['Dp Error'] = qcom_df['Delivery Val'] - qcom_df['Actuals Val']

# qcom_df['Dp Abs Error'] = np.abs(qcom_df['Dp Error'])



In [230]:
qcom_df.columns = qcom_df.columns.str.lower()
qcom_df

,month,depot,psku,brand,portfolio,m month,run_month,stat vol,channel,consensus vol,...,index rate,stat val,consensus val,actuals val,stat error,consensus error,stat abs error,consensus abs error,stat_bias,stat_bias_bucket
0,2026-04-30,D111,709567,SAFF OATS,Foods,M+1,2026-03-31,0.000000,ECOM,0.0,...,127515.619406,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0% to 5%
1,2026-04-30,D111,718288,SAFF GOLD,Saffola Oils,M+1,2026-03-31,0.000000,ECOM,0.0,...,138865.260689,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0% to 5%
2,2026-04-30,D111,718297,PCNO(R),CNO,M+1,2026-03-31,0.000000,ECOM,0.0,...,349274.001420,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0% to 5%
3,2026-04-30,D111,718299,PCNO(R),CNO,M+1,2026-03-31,0.000000,ECOM,0.0,...,349274.001420,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0% to 5%
4,2026-04-30,D111,718303,REV.LQDST,Others,M+1,2026-03-31,0.000000,ECOM,0.0,...,247926.608903,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0% to 5%
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
578929,2026-07-31,D677,810738,PABABY_GM,Skin Care,M+4,2026-03-31,13.968941,QCOM,0.0,...,366.484998,0.000512,0.0,0.0,0.000512,0.0,0.000512,0.0,0.0,0% to 5%
578930,2026-07-31,D677,810805,PABABY_GM,Skin Care,M+4,2026-03-31,0.000000,QCOM,0.0,...,366.484998,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0% to 5%
578931,2026-07-31,D677,810971,PA_ESS_HO,Hair Oils,M+4,2026-03-31,0.000000,QCOM,0.0,...,12860.631072,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0% to 5%
578932,2026-07-31,D677,811169,SW_SGPRF,Male Grooming,M+4,2026-03-31,0.000000,QCOM,0.0,...,1712.605337,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0% to 5%


In [111]:
# df = pd.read_excel("/data/aman_singh/acuuracy_check/acc_framework_feb.xlsx")
# df.columns = df.columns.str.lower()
# df = df[df['forecast_granularity']!='Depot x PSKU']

In [231]:
df.columns = df.columns.str.lower()

In [232]:
qcom_df.columns

Index(['month', 'depot', 'psku', 'brand', 'portfolio', 'm month', 'run_month',
       'stat vol', 'channel', 'consensus vol', 'actuals vol', 'index rate',
       'stat val', 'consensus val', 'actuals val', 'stat error',
       'consensus error', 'stat abs error', 'consensus abs error', 'stat_bias',
       'stat_bias_bucket'],
      dtype='object')

In [233]:

qcom_df['forecast_granularity'] = 'Depot x PSKU'
qcom_df['forecast_type'] = 'offtakes_to_primary'
#qcom_df['channel'] = 'QCOM'
qcom_df

,month,depot,psku,brand,portfolio,m month,run_month,stat vol,channel,consensus vol,...,consensus val,actuals val,stat error,consensus error,stat abs error,consensus abs error,stat_bias,stat_bias_bucket,forecast_granularity,forecast_type
0,2026-04-30,D111,709567,SAFF OATS,Foods,M+1,2026-03-31,0.000000,ECOM,0.0,...,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0% to 5%,Depot x PSKU,offtakes_to_primary
1,2026-04-30,D111,718288,SAFF GOLD,Saffola Oils,M+1,2026-03-31,0.000000,ECOM,0.0,...,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0% to 5%,Depot x PSKU,offtakes_to_primary
2,2026-04-30,D111,718297,PCNO(R),CNO,M+1,2026-03-31,0.000000,ECOM,0.0,...,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0% to 5%,Depot x PSKU,offtakes_to_primary
3,2026-04-30,D111,718299,PCNO(R),CNO,M+1,2026-03-31,0.000000,ECOM,0.0,...,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0% to 5%,Depot x PSKU,offtakes_to_primary
4,2026-04-30,D111,718303,REV.LQDST,Others,M+1,2026-03-31,0.000000,ECOM,0.0,...,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0% to 5%,Depot x PSKU,offtakes_to_primary
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
578929,2026-07-31,D677,810738,PABABY_GM,Skin Care,M+4,2026-03-31,13.968941,QCOM,0.0,...,0.0,0.0,0.000512,0.0,0.000512,0.0,0.0,0% to 5%,Depot x PSKU,offtakes_to_primary
578930,2026-07-31,D677,810805,PABABY_GM,Skin Care,M+4,2026-03-31,0.000000,QCOM,0.0,...,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0% to 5%,Depot x PSKU,offtakes_to_primary
578931,2026-07-31,D677,810971,PA_ESS_HO,Hair Oils,M+4,2026-03-31,0.000000,QCOM,0.0,...,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0% to 5%,Depot x PSKU,offtakes_to_primary
578932,2026-07-31,D677,811169,SW_SGPRF,Male Grooming,M+4,2026-03-31,0.000000,QCOM,0.0,...,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0% to 5%,Depot x PSKU,offtakes_to_primary


In [234]:
qcom_df = qcom_df[qcom_df['m month'] == 'M+1']

In [ ]:
# df2 = df.groupby(['channel', 'portfolio',
#        'brand', 'brand class', 'run_month', 'm month', 'month', 'depot',
#        'psku'])[['consensus vol', 'actuals vol', 'stat vol',
#        'stat val', 'consensus val', 'actuals val']].sum().reset_index()

In [ ]:
# df2[(df2['month'] == '2026-04-30') & (df2['channel']=='GT') & (df2['run_month']=='2026-03-31')]['actuals vol'].sum()
# # df2['actuals vol'].sum()

5101614.17

In [ ]:
# df2['stat error'] = df2['stat val'] - df2['actuals val']
# df2['consensus error'] = df2['consensus val'] - df2['actuals val']

# df2['stat abs error'] = np.abs(df2['stat error'])
# df2['consensus abs error'] = np.abs(df2['consensus error'])

In [ ]:
# df2[(df2['month'] == '2026-04-30') & (df2['channel']=='GT') & (df2['run_month']=='2026-03-31')]['stat val'].sum()

449.7762030932171

In [ ]:
# df2['forecast_granularity'] = 'Depot x PSKU'
# df2['forecast_type'] = 'secondary'

In [ ]:
# df2 = df2[df2['m month']=='M+1']

In [238]:
df.columns

Index(['forecast_granularity', 'forecast_type', 'channel', 'portfolio',
       'brand', 'run_month', 'm month', 'month', 'depot', 'psku', 'stat vol',
       'prophet vol', 'rf_vol', 'prophet heuristic vol', 'rf heuristic vol',
       'consensus vol', 'actuals vol', 'key', 'index rate', 'stat val',
       'consensus val', 'actuals val', 'stat error', 'consensus error',
       'stat abs error', 'consensus abs error', 'prophet vol_value',
       'rf_vol_value', 'prophet heuristic vol_value', 'rf heuristic vol_value',
       'stat_bias', 'stat_bias_bucket'],
      dtype='object')

In [242]:
df[(df['channel'] == 'MT') & (df['run_month'] == '2026-05-31') & (df['month'] == '2026-06-30')]['stat val'].sum()

127.83925921684045

In [243]:
final_df = pd.concat([df,qcom_df])
final_df

,forecast_granularity,forecast_type,channel,portfolio,brand,run_month,m month,month,depot,psku,...,stat error,consensus error,stat abs error,consensus abs error,prophet vol_value,rf_vol_value,prophet heuristic vol_value,rf heuristic vol_value,stat_bias,stat_bias_bucket
0,Depot x PSKU,secondary,B2B,0,JH_FRG_L,2026-05-31,M+1,2026-06-30,D117,734910,...,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0% to 5%
1,Depot x PSKU,secondary,B2B,0,JH_FRG_L,2026-05-31,M+1,2026-06-30,D117,734988,...,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0% to 5%
2,Depot x PSKU,secondary,B2B,0,JH_FRG_L,2026-05-31,M+1,2026-06-30,D117,734993,...,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0% to 5%
3,Depot x PSKU,secondary,B2B,0,JH_FRG_L,2026-05-31,M+1,2026-06-30,D117,734995,...,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0% to 5%
4,Depot x PSKU,secondary,B2B,0,JH_FRG_L,2026-05-31,M+1,2026-06-30,D117,735043,...,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0% to 5%
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
548236,Depot x PSKU,offtakes_to_primary,QCOM,Skin Care,PABABY_GM,2026-03-31,M+1,2026-04-30,D677,810738,...,0.000509,0.000000,0.000509,0.000000,NaN,NaN,NaN,NaN,0.0,0% to 5%
548237,Depot x PSKU,offtakes_to_primary,QCOM,Skin Care,PABABY_GM,2026-03-31,M+1,2026-04-30,D677,810805,...,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,0.0,0% to 5%
548238,Depot x PSKU,offtakes_to_primary,QCOM,Hair Oils,PA_ESS_HO,2026-03-31,M+1,2026-04-30,D677,810971,...,-0.000926,-0.000926,0.000926,0.000926,NaN,NaN,NaN,NaN,-1.0,< -15%
548239,Depot x PSKU,offtakes_to_primary,QCOM,Male Grooming,SW_SGPRF,2026-03-31,M+1,2026-04-30,D677,811169,...,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,0.0,0% to 5%


In [681]:
# final_df[final_df['month'] == '2026-02-28'][['channel', 'asm', 'depot', 'psku', 'month', 'stat vol', 'consensus vol', 'actuals vol']].head(60)

In [244]:
final_df

,forecast_granularity,forecast_type,channel,portfolio,brand,run_month,m month,month,depot,psku,...,stat error,consensus error,stat abs error,consensus abs error,prophet vol_value,rf_vol_value,prophet heuristic vol_value,rf heuristic vol_value,stat_bias,stat_bias_bucket
0,Depot x PSKU,secondary,B2B,0,JH_FRG_L,2026-05-31,M+1,2026-06-30,D117,734910,...,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0% to 5%
1,Depot x PSKU,secondary,B2B,0,JH_FRG_L,2026-05-31,M+1,2026-06-30,D117,734988,...,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0% to 5%
2,Depot x PSKU,secondary,B2B,0,JH_FRG_L,2026-05-31,M+1,2026-06-30,D117,734993,...,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0% to 5%
3,Depot x PSKU,secondary,B2B,0,JH_FRG_L,2026-05-31,M+1,2026-06-30,D117,734995,...,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0% to 5%
4,Depot x PSKU,secondary,B2B,0,JH_FRG_L,2026-05-31,M+1,2026-06-30,D117,735043,...,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0% to 5%
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
548236,Depot x PSKU,offtakes_to_primary,QCOM,Skin Care,PABABY_GM,2026-03-31,M+1,2026-04-30,D677,810738,...,0.000509,0.000000,0.000509,0.000000,NaN,NaN,NaN,NaN,0.0,0% to 5%
548237,Depot x PSKU,offtakes_to_primary,QCOM,Skin Care,PABABY_GM,2026-03-31,M+1,2026-04-30,D677,810805,...,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,0.0,0% to 5%
548238,Depot x PSKU,offtakes_to_primary,QCOM,Hair Oils,PA_ESS_HO,2026-03-31,M+1,2026-04-30,D677,810971,...,-0.000926,-0.000926,0.000926,0.000926,NaN,NaN,NaN,NaN,-1.0,< -15%
548239,Depot x PSKU,offtakes_to_primary,QCOM,Male Grooming,SW_SGPRF,2026-03-31,M+1,2026-04-30,D677,811169,...,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,0.0,0% to 5%


In [245]:
qcom_df['run_month'].unique()

<DatetimeArray>
['2026-03-31 00:00:00', '2026-04-30 00:00:00', '2026-05-31 00:00:00',
 '2025-11-30 00:00:00', '2025-12-31 00:00:00', '2026-01-31 00:00:00',
 '2026-02-28 00:00:00']
Length: 7, dtype: datetime64[ns]

In [ ]:
# final_df[final_df['run_month'].isin(['2026-04-30']) ].to_excel('/data/aman_singh/acuuracy_check/acc_framework_may_final.xlsx')

In [ ]:
# x = final_df[final_df['run_month'].isin(['2026-04-30']) ]
# x

NameError: name 'final_df' is not defined

In [ ]:
# final_df[(final_df['month'] >= '2026-01-31') & (final_df['month'] <= '2026-02-28')]['channel'].unique()

array(['B2B', 'ECOM', 'GT', 'MT', 'Qcom', 'Ecom'], dtype=object)

### offtakes

In [246]:
query = """select * from TRN_MIL_DF_OFFTAKES_OUTPUT"""
df_offtakes = pd.read_sql(query, dev_conn)
df_offtakes.columns = df_offtakes.columns.str.lower()
df_offtakes['run_month'] = pd.to_datetime(df_offtakes['run_month'])
df_offtakes['month_date'] = pd.to_datetime(df_offtakes['month_date'])
df_offtakes

,month_date,platform_name,parent_material_code,brand_code,portfolio,run_month,m month,pred_prophet,pred_rf,final_heuristic_prophet_value_2,channel
0,2026-04-30,Amazon ARIPL,718288,SAFF GOLD,Saffola Oils,2026-04-30,M,28.076909,27.69528,0.421023,ECOM
1,2026-04-30,Amazon ARIPL,718321,SAFF KO,Saffola Oils,2026-04-30,M,0.000000,0.00000,0.000000,ECOM
2,2026-04-30,Amazon ARIPL,718322,SAFF KO,Saffola Oils,2026-04-30,M,7.161035,9.86000,0.188237,ECOM
3,2026-04-30,Amazon ARIPL,718323,SF_IMV_MK,Foods,2026-04-30,M,0.000000,0.00000,0.000000,ECOM
4,2026-04-30,Amazon ARIPL,718328,SAFF KOCO,Saffola Oils,2026-04-30,M,3.897973,6.99084,0.113964,ECOM
...,...,...,...,...,...,...,...,...,...,...,...
129460,2026-12-31,Nykaa,810584,KAYA_ML,Skin Care,2026-03-31,M+9,0.000000,0.00000,0.000000,ECOM
129461,2026-12-31,Nykaa,810673,PA_ESS_HO,Hair Oils,2026-03-31,M+9,0.000000,0.00000,0.002372,ECOM
129462,2026-12-31,Nykaa,810674,PA_ESS_HO,Hair Oils,2026-03-31,M+9,0.000000,0.00000,0.000524,ECOM
129463,2026-12-31,Nykaa,810738,PABABY_GM,Skin Care,2026-03-31,M+9,0.000000,0.00000,0.000000,ECOM


In [247]:
df_offtakes['run_month'].unique()

<DatetimeArray>
['2026-04-30 00:00:00', '2026-01-31 00:00:00', '2026-02-28 00:00:00',
 '2026-03-31 00:00:00']
Length: 4, dtype: datetime64[ns]

In [7]:
query_qcom = """select * from TRN_DF_QCOM_OFFTAKE_CHAIN_PSKU
where run_month = '2026-06-30'
and month_date between '2026-01-31' and '2026-05-31'
"""
qcom_df = pd.read_sql(query_qcom, dev_conn)
qcom_df.columns = qcom_df.columns.str.lower()
qcom_df['run_month'] = pd.to_datetime(qcom_df['run_month'])
qcom_df['month_date'] = pd.to_datetime(qcom_df['month_date'])
qcom_df['channel'] = 'QCOM'
qcom_df

,month_date,key,platform_name,parent_material_code,brand_code,vol_in_rum,run_month,imputed,channel
0,2026-01-31,blinkit_718288,blinkit,718288.0,SAFF GOLD,62.4780,2026-06-30,0,QCOM
1,2026-02-28,blinkit_718288,blinkit,718288.0,SAFF GOLD,60.5760,2026-06-30,0,QCOM
2,2026-03-31,blinkit_718288,blinkit,718288.0,SAFF GOLD,77.6460,2026-06-30,0,QCOM
3,2026-04-30,blinkit_718288,blinkit,718288.0,SAFF GOLD,47.6940,2026-06-30,0,QCOM
4,2026-01-31,blinkit_718310,blinkit,718310.0,PCNO(R),0.0010,2026-06-30,0,QCOM
...,...,...,...,...,...,...,...,...,...
4355,2026-05-31,zepto_810685,zepto,810685.0,SAF-MUSLI,0.0016,2026-06-30,0,QCOM
4356,2026-05-31,zepto_810738,zepto,810738.0,PABABY_GM,48.5080,2026-06-30,0,QCOM
4357,2026-05-31,zepto_810971,zepto,810971.0,PA_ESS_HO,0.6300,2026-06-30,0,QCOM
4358,2026-05-31,zepto_811005,zepto,811005.0,PA_ESS_HO,0.7140,2026-06-30,0,QCOM


In [8]:
query_ecom = """select * from TRN_DF_ECOM_OFFTAKE_CHAIN_PSKU
where run_month = '2026-06-30'
and month_date between '2026-01-31' and '2026-05-31'
"""
ecom_df = pd.read_sql(query_ecom, dev_conn)
ecom_df.columns = ecom_df.columns.str.lower()
ecom_df['run_month'] = pd.to_datetime(ecom_df['run_month'])
ecom_df['month_date'] = pd.to_datetime(ecom_df['month_date'])
ecom_df['channel'] = 'ECOM'
ecom_df

,month_date,platform_name,parent_material_code,brand_code,vol_in_rum,indexbpm,imputed,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,run_month,channel
0,2026-01-31,Amazon ARIPL,718288,SAFF GOLD,22.242,30.618990,0,0,0,0,0,0,0,0,0,0,0,2026-06-30,ECOM
1,2026-02-28,Amazon ARIPL,718288,SAFF GOLD,20.178,27.777630,0,0,0,0,0,0,0,0,0,0,0,2026-06-30,ECOM
2,2026-03-31,Amazon ARIPL,718288,SAFF GOLD,39.372,54.200650,0,0,0,0,0,0,0,0,0,0,0,2026-06-30,ECOM
3,2026-04-30,Amazon ARIPL,718288,SAFF GOLD,12.102,16.805470,0,0,0,0,0,0,0,0,0,0,0,2026-06-30,ECOM
4,2026-05-31,Amazon ARIPL,718288,SAFF GOLD,30.204,41.942863,0,0,0,0,0,0,0,0,0,0,0,2026-06-30,ECOM
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15804,2026-02-28,Nykaa,810805,PABABY_GM,11.946,0.053890,0,0,0,0,0,0,0,0,0,0,0,2026-06-30,ECOM
15805,2026-03-31,Nykaa,810805,PABABY_GM,18.462,0.083290,0,0,0,0,0,0,0,0,0,0,0,2026-06-30,ECOM
15806,2026-04-30,Nykaa,810805,PABABY_GM,26.064,0.095520,0,0,0,0,0,0,0,0,0,0,0,2026-06-30,ECOM
15807,2026-05-31,Nykaa,810805,PABABY_GM,0.000,0.000000,1,0,0,0,0,0,0,0,0,0,0,2026-06-30,ECOM


In [9]:
offtake_actuals = pd.concat([qcom_df, ecom_df], ignore_index=True)
offtake_actuals

,month_date,key,platform_name,parent_material_code,brand_code,vol_in_rum,run_month,imputed,channel,indexbpm,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2
0,2026-01-31,blinkit_718288,blinkit,718288.0,SAFF GOLD,62.478,2026-06-30,0,QCOM,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-02-28,blinkit_718288,blinkit,718288.0,SAFF GOLD,60.576,2026-06-30,0,QCOM,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-03-31,blinkit_718288,blinkit,718288.0,SAFF GOLD,77.646,2026-06-30,0,QCOM,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-04-30,blinkit_718288,blinkit,718288.0,SAFF GOLD,47.694,2026-06-30,0,QCOM,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-01-31,blinkit_718310,blinkit,718310.0,PCNO(R),0.001,2026-06-30,0,QCOM,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20164,2026-02-28,NaN,Nykaa,810805.0,PABABY_GM,11.946,2026-06-30,0,ECOM,0.053890,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
20165,2026-03-31,NaN,Nykaa,810805.0,PABABY_GM,18.462,2026-06-30,0,ECOM,0.083290,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
20166,2026-04-30,NaN,Nykaa,810805.0,PABABY_GM,26.064,2026-06-30,0,ECOM,0.095520,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
20167,2026-05-31,NaN,Nykaa,810805.0,PABABY_GM,0.000,2026-06-30,1,ECOM,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [30]:
df_offtakes['run_month'].unique()

<DatetimeArray>
['2026-04-30 00:00:00', '2026-01-31 00:00:00', '2026-02-28 00:00:00',
 '2026-03-31 00:00:00']
Length: 4, dtype: datetime64[ns]

In [31]:
# df_offtakes = df_offtakes[df_offtakes['run_month'].isin(['2026-02-28', '2026-01-31'])]
df_offtakes = df_offtakes[df_offtakes['m month'] == 'M+1']
df_offtakes

,month_date,platform_name,parent_material_code,brand_code,portfolio,run_month,m month,pred_prophet,pred_rf,final_heuristic_prophet_value_2,channel
3246,2026-05-31,Amazon ARIPL,718288,SAFF GOLD,Saffola Oils,2026-04-30,M+1,29.403313,27.76920,0.421023,ECOM
3247,2026-05-31,Amazon ARIPL,718321,SAFF KO,Saffola Oils,2026-04-30,M+1,0.000000,0.00000,0.000000,ECOM
3248,2026-05-31,Amazon ARIPL,718322,SAFF KO,Saffola Oils,2026-04-30,M+1,7.271279,16.05050,0.187607,ECOM
3249,2026-05-31,Amazon ARIPL,718323,SF_IMV_MK,Foods,2026-04-30,M+1,0.000000,0.00000,0.000000,ECOM
3250,2026-05-31,Amazon ARIPL,718328,SAFF KOCO,Saffola Oils,2026-04-30,M+1,3.942144,6.99084,0.113964,ECOM
...,...,...,...,...,...,...,...,...,...,...,...
109322,2026-04-30,Nykaa,810605,KAYA_ML,Skin Care,2026-03-31,M+1,0.000000,0.00000,0.000000,ECOM
109323,2026-04-30,Nykaa,810673,PA_ESS_HO,Hair Oils,2026-03-31,M+1,0.000000,0.00000,0.002372,ECOM
109324,2026-04-30,Nykaa,810674,PA_ESS_HO,Hair Oils,2026-03-31,M+1,0.000000,0.00000,0.000524,ECOM
109325,2026-04-30,Nykaa,810738,PABABY_GM,Skin Care,2026-03-31,M+1,0.000000,0.00000,0.000000,ECOM


In [32]:
df_offtakes

,month_date,platform_name,parent_material_code,brand_code,portfolio,run_month,m month,pred_prophet,pred_rf,final_heuristic_prophet_value_2,channel
3246,2026-05-31,Amazon ARIPL,718288,SAFF GOLD,Saffola Oils,2026-04-30,M+1,29.403313,27.76920,0.421023,ECOM
3247,2026-05-31,Amazon ARIPL,718321,SAFF KO,Saffola Oils,2026-04-30,M+1,0.000000,0.00000,0.000000,ECOM
3248,2026-05-31,Amazon ARIPL,718322,SAFF KO,Saffola Oils,2026-04-30,M+1,7.271279,16.05050,0.187607,ECOM
3249,2026-05-31,Amazon ARIPL,718323,SF_IMV_MK,Foods,2026-04-30,M+1,0.000000,0.00000,0.000000,ECOM
3250,2026-05-31,Amazon ARIPL,718328,SAFF KOCO,Saffola Oils,2026-04-30,M+1,3.942144,6.99084,0.113964,ECOM
...,...,...,...,...,...,...,...,...,...,...,...
109322,2026-04-30,Nykaa,810605,KAYA_ML,Skin Care,2026-03-31,M+1,0.000000,0.00000,0.000000,ECOM
109323,2026-04-30,Nykaa,810673,PA_ESS_HO,Hair Oils,2026-03-31,M+1,0.000000,0.00000,0.002372,ECOM
109324,2026-04-30,Nykaa,810674,PA_ESS_HO,Hair Oils,2026-03-31,M+1,0.000000,0.00000,0.000524,ECOM
109325,2026-04-30,Nykaa,810738,PABABY_GM,Skin Care,2026-03-31,M+1,0.000000,0.00000,0.000000,ECOM


In [33]:
offtake_actuals['channel'].unique()

array(['QCOM', 'ECOM'], dtype=object)

In [34]:
df_offtakes['channel'].unique()

array(['ECOM', 'QCOM'], dtype=object)

In [35]:
df_offtakes

,month_date,platform_name,parent_material_code,brand_code,portfolio,run_month,m month,pred_prophet,pred_rf,final_heuristic_prophet_value_2,channel
3246,2026-05-31,Amazon ARIPL,718288,SAFF GOLD,Saffola Oils,2026-04-30,M+1,29.403313,27.76920,0.421023,ECOM
3247,2026-05-31,Amazon ARIPL,718321,SAFF KO,Saffola Oils,2026-04-30,M+1,0.000000,0.00000,0.000000,ECOM
3248,2026-05-31,Amazon ARIPL,718322,SAFF KO,Saffola Oils,2026-04-30,M+1,7.271279,16.05050,0.187607,ECOM
3249,2026-05-31,Amazon ARIPL,718323,SF_IMV_MK,Foods,2026-04-30,M+1,0.000000,0.00000,0.000000,ECOM
3250,2026-05-31,Amazon ARIPL,718328,SAFF KOCO,Saffola Oils,2026-04-30,M+1,3.942144,6.99084,0.113964,ECOM
...,...,...,...,...,...,...,...,...,...,...,...
109322,2026-04-30,Nykaa,810605,KAYA_ML,Skin Care,2026-03-31,M+1,0.000000,0.00000,0.000000,ECOM
109323,2026-04-30,Nykaa,810673,PA_ESS_HO,Hair Oils,2026-03-31,M+1,0.000000,0.00000,0.002372,ECOM
109324,2026-04-30,Nykaa,810674,PA_ESS_HO,Hair Oils,2026-03-31,M+1,0.000000,0.00000,0.000524,ECOM
109325,2026-04-30,Nykaa,810738,PABABY_GM,Skin Care,2026-03-31,M+1,0.000000,0.00000,0.000000,ECOM


In [36]:
df_offtakes['platform_name'] = df_offtakes['platform_name'].str.lower()
offtake_actuals['platform_name'] = offtake_actuals['platform_name'].str.lower()
df_offtakes['parent_material_code'] = df_offtakes['parent_material_code'].astype(int)
offtake_actuals['parent_material_code'] = offtake_actuals['parent_material_code'].astype(int)

In [37]:
offtake_actuals[['channel', 'platform_name', 'parent_material_code','month_date', 'vol_in_rum']].dtypes

channel                         object
platform_name                   object
parent_material_code             int64
month_date              datetime64[ns]
vol_in_rum                     float64
dtype: object

In [39]:
df_offtakes[['channel', 'platform_name', 'parent_material_code','month_date']].dtypes

channel                         object
platform_name                   object
parent_material_code             int64
month_date              datetime64[ns]
dtype: object

In [40]:
x = df_offtakes.copy()

In [51]:
df_offtakes = x.copy()

In [52]:
df_offtakes = df_offtakes.merge(
    offtake_actuals[['channel', 'platform_name', 'parent_material_code','month_date', 'vol_in_rum']]
    ,
    on=['channel', 'platform_name', 'parent_material_code','month_date'],
    how='left'
)

In [53]:
df_offtakes.rename(columns={'brand_code':'brand'}, inplace=True)

In [54]:
def read_qtr_ind_rate_table():
    query = """select * from DWH_SAP_INDEX_TURNOVER_MONTHWISE 
                where latest_rate_flag=1 and company_code='MIL'"""
    qtr_ind_rate_data = pd.read_sql(con=prod_conn, sql=query)
    qtr_ind_rate_data.columns = qtr_ind_rate_data.columns.str.lower()
    qtr_ind_rate =  qtr_ind_rate_data[['date', 'brand_code', 'turnover']]
    qtr_ind_rate = qtr_ind_rate.rename(columns= {'date':'month_date', 'turnover':'qtr_ind_rate'})
    
    return qtr_ind_rate

qtr_ind_rate_df = read_qtr_ind_rate_table()
qtr_ind_rate_df.head()
len_before_merge = len(df_offtakes)
df_offtakes = df_offtakes.merge(
    qtr_ind_rate_df.drop('month_date', axis=1).rename(
        columns={'brand_code': 'brand', 'qtr_ind_rate': 'Index Rate'}
    ),
    on=['brand'], 
    how='left'
)
assert len_before_merge == len(df_offtakes)
del len_before_merge

In [55]:
df_offtakes.rename(columns={'month_date':'month', 'final_heuristic_prophet_value_2':'offtakes_forecasted_value'}, inplace=True)
df_offtakes['offtakes_forecasted_vol'] = df_offtakes['offtakes_forecasted_value'] * (10 ** 7) / df_offtakes['Index Rate']
df_offtakes['offtakes actuals val'] = df_offtakes['vol_in_rum'] * df_offtakes['Index Rate'] / (10 ** 7)
df_offtakes['offtakes error'] = df_offtakes['offtakes actuals val'] - df_offtakes['offtakes_forecasted_value'] 
df_offtakes['offtakes abs error'] = np.abs(df_offtakes['offtakes error'])
df_offtakes


,month,platform_name,parent_material_code,brand,portfolio,run_month,m month,pred_prophet,pred_rf,offtakes_forecasted_value,channel,vol_in_rum,Index Rate,offtakes_forecasted_vol,offtakes actuals val,offtakes error,offtakes abs error
0,2026-05-31,amazon aripl,718288,SAFF GOLD,Saffola Oils,2026-04-30,M+1,29.403313,27.76920,0.421023,ECOM,30.2040,138865.260689,30.318787,0.419429,-0.001594,0.001594
1,2026-05-31,amazon aripl,718321,SAFF KO,Saffola Oils,2026-04-30,M+1,0.000000,0.00000,0.000000,ECOM,0.0000,168827.536176,0.000000,0.000000,0.000000,0.000000
2,2026-05-31,amazon aripl,718322,SAFF KO,Saffola Oils,2026-04-30,M+1,7.271279,16.05050,0.187607,ECOM,12.1500,168827.536176,11.112333,0.205125,0.017519,0.017519
3,2026-05-31,amazon aripl,718323,SF_IMV_MK,Foods,2026-04-30,M+1,0.000000,0.00000,0.000000,ECOM,0.0000,739265.904412,0.000000,0.000000,0.000000,0.000000
4,2026-05-31,amazon aripl,718328,SAFF KOCO,Saffola Oils,2026-04-30,M+1,3.942144,6.99084,0.113964,ECOM,9.6561,123636.889888,9.217630,0.119385,0.005421,0.005421
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16426,2026-04-30,nykaa,810605,KAYA_ML,Skin Care,2026-03-31,M+1,0.000000,0.00000,0.000000,ECOM,0.0000,1226.374229,0.000000,0.000000,0.000000,0.000000
16427,2026-04-30,nykaa,810673,PA_ESS_HO,Hair Oils,2026-03-31,M+1,0.000000,0.00000,0.002372,ECOM,1.7640,12860.631072,1.844295,0.002269,-0.000103,0.000103
16428,2026-04-30,nykaa,810674,PA_ESS_HO,Hair Oils,2026-03-31,M+1,0.000000,0.00000,0.000524,ECOM,0.5880,12860.631072,0.407243,0.000756,0.000232,0.000232
16429,2026-04-30,nykaa,810738,PABABY_GM,Skin Care,2026-03-31,M+1,0.000000,0.00000,0.000000,ECOM,0.0000,366.484998,0.000000,0.000000,0.000000,0.000000


In [58]:
df_offtakes[df_offtakes['channel']=='QCOM'].groupby(['month'])['offtakes actuals val'].sum()

month
2026-02-28    29.097719
2026-03-31    34.160361
2026-04-30    27.965569
2026-05-31    30.909090
Name: offtakes actuals val, dtype: float64

In [59]:
df_offtakes['offtakes_bias'] = df_offtakes['offtakes error']/df_offtakes['offtakes actuals val']
df_offtakes = df_offtakes.fillna(0)

df_offtakes['offtakes_bias'] = (
    df_offtakes['offtakes_bias']
    .replace([np.inf, -np.inf], 0)
    .fillna(0)
)

bins = [-np.inf, -0.15, -0.10, -0.05, 0, 0.05, 0.10, 0.15, np.inf]
labels = [
    '< -15%',
    '-15% to -10%',
    '-10% to -5%',
    '-5% to 0%',
    '0% to 5%',
    '5% to 10%',
    '10% to 15%',
    '> 15%'
]

df_offtakes['offtakes_bias_bucket'] = pd.cut(
    df_offtakes['offtakes_bias'],
    bins=bins,
    labels=labels,
    right=False  
)


In [60]:
df_offtakes['forecast_granularity'] = 'Chain x PSKU'
df_offtakes['forecast_type'] = 'offtakes'

In [61]:
df_offtakes.to_csv('acc_offtakes_till_may.csv')

In [26]:
df_stat_fva = pd.read_excel('/data/aman_singh/acuuracy_check/fva_and_stat_accuracy.xlsx', sheet_name = 'fva_depot_psku_final')
df_stat_fva

,forecast_granularity,forecast_type,channel,depot,psku,brand,portfolio,month,m month,stat vol,...,dp error drm,dp abs error drm,dp error cam,dp abs error cam,dp error bam,dp abs error bam,Bias bucket drm,bias bucket cam,bias_bucket_bam,bias_bucket_plan
0,Depot x PSKU,secondary,B2B,D112,718287,PCNO(R),CNO,2025-11-30,M+1,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,5 | 0% to 5%,5 | 0% to 5%,5 | 0% to 5%,5 | 0% to 5%
1,Depot x PSKU,secondary,B2B,D112,718287,PCNO(R),CNO,2025-12-31,M+1,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,5 | 0% to 5%,5 | 0% to 5%,5 | 0% to 5%,5 | 0% to 5%
2,Depot x PSKU,secondary,B2B,D112,718297,PCNO(R),CNO,2025-11-30,M+1,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,5 | 0% to 5%,5 | 0% to 5%,5 | 0% to 5%,5 | 0% to 5%
3,Depot x PSKU,secondary,B2B,D112,718297,PCNO(R),CNO,2025-12-31,M+1,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,5 | 0% to 5%,5 | 0% to 5%,5 | 0% to 5%,5 | 0% to 5%
4,Depot x PSKU,secondary,B2B,D112,718297,PCNO(R),CNO,2026-01-31,M+1,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,5 | 0% to 5%,5 | 0% to 5%,5 | 0% to 5%,5 | 0% to 5%
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
142881,Depot x PSKU,offtakes_to_primary,QCOM,D677,810521,SAF_CDPRS,Saffola Oils,2026-03-31,M+1,0.012889,...,0.0,0.0,0.0,0.0,0.0,0.0,5 | 0% to 5%,5 | 0% to 5%,5 | 0% to 5%,5 | 0% to 5%
142882,Depot x PSKU,offtakes_to_primary,QCOM,D677,810522,SAF_CDPRS,Saffola Oils,2026-03-31,M+1,0.012509,...,0.0,0.0,0.0,0.0,0.0,0.0,5 | 0% to 5%,5 | 0% to 5%,5 | 0% to 5%,5 | 0% to 5%
142883,Depot x PSKU,offtakes_to_primary,QCOM,D677,810738,PABABY_GM,Skin Care,2026-03-31,M+1,11.106341,...,0.0,0.0,0.0,0.0,0.0,0.0,5 | 0% to 5%,5 | 0% to 5%,5 | 0% to 5%,5 | 0% to 5%
142884,Depot x PSKU,offtakes_to_primary,QCOM,D677,810805,PABABY_GM,Skin Care,2026-03-31,M+1,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,5 | 0% to 5%,5 | 0% to 5%,5 | 0% to 5%,5 | 0% to 5%


In [28]:
df_offtakes.rename(columns={'platform_name':'chain', 'parent_material_code':'psku'
                            ,'Index Rate':'index rate'}, inplace=True)
df_offtakes.drop(['vol_in_rum','pred_prophet', 'pred_rf','run_month'], axis=1, inplace=True)
df_offtakes

,month,chain,psku,brand,portfolio,m month,offtakes_forecasted_value,channel,index rate,offtakes_forecasted_vol,offtakes actuals val,offtakes error,offtakes abs error,offtakes_bias,offtakes_bias_bucket,forecast_granularity,forecast_type
0,2026-02-28,Blinkit,718288,SAFF GOLD,Saffola Oils,M+1,0.829005,QCOM,138865.260689,59.698539,0.841190,0.012185,0.012185,0.014485,0% to 5%,Chain x PSKU,offtakes
1,2026-02-28,Blinkit,718310,PCNO(R),CNO,M+1,0.000790,QCOM,349274.001420,0.022616,0.000017,-0.000772,0.000772,-44.231134,< -15%,Chain x PSKU,offtakes
2,2026-02-28,Blinkit,718312,PCNO(R),CNO,M+1,0.281155,QCOM,349274.001420,8.049687,0.194406,-0.086749,0.086749,-0.446225,< -15%,Chain x PSKU,offtakes
3,2026-02-28,Blinkit,718315,PCNO(R),CNO,M+1,0.000000,QCOM,349274.001420,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%,Chain x PSKU,offtakes
4,2026-02-28,Blinkit,718317,H&C,Hair Oils,M+1,0.000000,QCOM,388.076436,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%,Chain x PSKU,offtakes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8297,2026-03-31,Nykaa,810605,KAYA_ML,Skin Care,M+1,0.000000,ECOM,1226.374229,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%,Chain x PSKU,offtakes
8298,2026-03-31,Nykaa,810673,PA_ESS_HO,Hair Oils,M+1,0.002456,ECOM,12860.631072,1.909829,0.002269,-0.000188,0.000188,-0.082669,-10% to -5%,Chain x PSKU,offtakes
8299,2026-03-31,Nykaa,810674,PA_ESS_HO,Hair Oils,M+1,0.000933,ECOM,12860.631072,0.725548,0.000522,-0.000411,0.000411,-0.787063,< -15%,Chain x PSKU,offtakes
8300,2026-03-31,Nykaa,810738,PABABY_GM,Skin Care,M+1,0.000000,ECOM,366.484998,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%,Chain x PSKU,offtakes


In [29]:
final_df = pd.concat([df_stat_fva, df_offtakes], ignore_index=True)
final_df

,forecast_granularity,forecast_type,channel,depot,psku,brand,portfolio,month,m month,stat vol,...,bias_bucket_bam,bias_bucket_plan,chain,offtakes_forecasted_value,offtakes_forecasted_vol,offtakes actuals val,offtakes error,offtakes abs error,offtakes_bias,offtakes_bias_bucket
0,Depot x PSKU,secondary,B2B,D112,718287,PCNO(R),CNO,2025-11-30,M+1,0.0,...,5 | 0% to 5%,5 | 0% to 5%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Depot x PSKU,secondary,B2B,D112,718287,PCNO(R),CNO,2025-12-31,M+1,0.0,...,5 | 0% to 5%,5 | 0% to 5%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Depot x PSKU,secondary,B2B,D112,718297,PCNO(R),CNO,2025-11-30,M+1,0.0,...,5 | 0% to 5%,5 | 0% to 5%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Depot x PSKU,secondary,B2B,D112,718297,PCNO(R),CNO,2025-12-31,M+1,0.0,...,5 | 0% to 5%,5 | 0% to 5%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Depot x PSKU,secondary,B2B,D112,718297,PCNO(R),CNO,2026-01-31,M+1,0.0,...,5 | 0% to 5%,5 | 0% to 5%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
151183,Chain x PSKU,offtakes,ECOM,NaN,810605,KAYA_ML,Skin Care,2026-03-31,M+1,NaN,...,NaN,NaN,Nykaa,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%
151184,Chain x PSKU,offtakes,ECOM,NaN,810673,PA_ESS_HO,Hair Oils,2026-03-31,M+1,NaN,...,NaN,NaN,Nykaa,0.002456,1.909829,0.002269,-0.000188,0.000188,-0.082669,-10% to -5%
151185,Chain x PSKU,offtakes,ECOM,NaN,810674,PA_ESS_HO,Hair Oils,2026-03-31,M+1,NaN,...,NaN,NaN,Nykaa,0.000933,0.725548,0.000522,-0.000411,0.000411,-0.787063,< -15%
151186,Chain x PSKU,offtakes,ECOM,NaN,810738,PABABY_GM,Skin Care,2026-03-31,M+1,NaN,...,NaN,NaN,Nykaa,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%


In [30]:
final_df.columns

Index(['forecast_granularity', 'forecast_type', 'channel', 'depot', 'psku',
       'brand', 'portfolio', 'month', 'm month', 'stat vol', 'actuals vol',
       'consensus vol', 'index rate', 'stat val', 'consensus val',
       'actuals val', 'stat error', 'consensus error', 'stat abs error',
       'consensus abs error', 'stat_bias', 'stat_bias_bucket',
       'delivery_vol_drm', 'delivery val_drm', 'delivery_vol_cam',
       'delivery val_cam', 'delivery_vol_bam', 'delivery val_bam',
       'brand class', 'dp error drm', 'dp abs error drm', 'dp error cam',
       'dp abs error cam', 'dp error bam', 'dp abs error bam',
       'Bias bucket drm', 'bias bucket cam', 'bias_bucket_bam',
       'bias_bucket_plan', 'chain', 'offtakes_forecasted_value',
       'offtakes_forecasted_vol', 'offtakes actuals val', 'offtakes error',
       'offtakes abs error', 'offtakes_bias', 'offtakes_bias_bucket'],
      dtype='object')

In [32]:
final_df.to_csv('/data/aman_singh/acuuracy_check/fva_stat_offtake_accuracy.csv', index=False)

### Chain psku primary accuracy

In [64]:
qcom_df = pd.read_sql(
    """select * from TRN_MIL_DF_OFT2PRIM_CPSKU""",
    dev_conn
)
qcom_df.columns = qcom_df.columns.str.lower()
qcom_df['run_month'] = pd.to_datetime(qcom_df['run_month'])
qcom_df['month'] = pd.to_datetime(qcom_df['month'])

In [65]:
qcom_df#[qcom_df['Channel']=='Ecom']

,month,chain,psku,brand,portfolio,m month,run_month,calculated primary vol,channel
0,2025-12-31,Blinkit,718287,PCNO(R),CNO,M,2025-12-31,0.000000,Qcom
1,2025-12-31,Blinkit,718288,SAFF GOLD,Saffola Oils,M,2025-12-31,76.619500,Qcom
2,2025-12-31,Blinkit,718297,PCNO(R),CNO,M,2025-12-31,0.000000,Qcom
3,2025-12-31,Blinkit,718299,PCNO(R),CNO,M,2025-12-31,0.000000,Qcom
4,2025-12-31,Blinkit,718300,PCNO FLEX,CNO,M,2025-12-31,0.000000,Qcom
...,...,...,...,...,...,...,...,...,...
91428,2026-08-31,Zepto,810971,PA_ESS_HO,Hair Oils,M+4,2026-04-30,0.180968,Qcom
91429,2026-08-31,Zepto,811005,PA_ESS_HO,Hair Oils,M+4,2026-04-30,0.154828,Qcom
91430,2026-08-31,Zepto,811169,SW_SGPRF,Male Grooming,M+4,2026-04-30,0.000000,Qcom
91431,2026-08-31,Zepto,811181,SAF_CDPRS,Saffola Oils,M+4,2026-04-30,0.043734,Qcom


In [66]:
realignment_df = pd.read_sql(
    """select * from trn_mil_asm_psku_realignment""",
    dev_conn
)
realignment_df.columns = realignment_df.columns.str.lower()

# realignment_df = realignment_df[
#     realignment_df['channel'].isin(['QCOM', 'QCOM B2C', 'ALL'])]


def realign_pskus(data, column, channel='QCOM'):
    realignment_data = realignment_df.copy()
    realignment_data = realignment_data[
        realignment_data['channel'].isin([channel, channel + ' B2C', 'ALL'])
    ]

    assert realignment_data['psku old'].dtype == 'int64'
    assert realignment_data['psku new'].dtype == 'int64'
    realignment_data = realignment_data[['psku old', 'psku new']].drop_duplicates()
    realignment_data = realignment_data.set_index('psku old').to_dict()['psku new']

    
    data[column] = data[column].astype(int)

    for old_psku, new_psku in realignment_data.items():
        data.loc[
            data[column] == old_psku, column
        ] = new_psku

    return data

In [67]:
chain_psku_primary_query = """
SELECT
    CASE
        WHEN MCM.chain = 'Kiranakart Technologies' THEN 'Zepto'
        WHEN MCM.chain = 'Grofers' THEN 'Blinkit'
        ELSE MCM.chain
    END AS chain,
    MM.parent_material_code,
    MM.material_group_code,
    LAST_DAY(MESR.month_date) AS month_date,
    SUM(pri_actuals_vol_rum) AS pri_actuals_vol_rum,
    SUM(pri_apo_plan_vol_rum) AS pri_apo_plan_vol_rum,
    SUM(MESR.sec_apo_plan_vol_rum) AS sec_apo_plan_vol_rum,
    SUM(MESR.sec_actuals_vol_rum) AS sec_actuals_vol_rum
FROM
    dwh_bpm_dist_sku_daily MESR
JOIN 
(
    SELECT
        customer,
        chain_type,
        chain
    FROM
        mst_chain_master
    WHERE
        chain_type = 'E Com B2C' AND 
        chain IN ('Grofers', 'Zepto', 'Kiranakart Technologies', 'Swiggy', 'ZEPTO')
) MCM ON MESR.distributor_code = MCM.customer
JOIN
(
    SELECT
        material_code,
        parent_material_code,
        material_group_code,
        uom_reporting,
        vol_per_unit
    FROM 
        mst_material
    WHERE
        company_code='MIL' AND
        latest_record_ind=1
) MM ON MESR.material_code = MM.material_code
JOIN
(
    SELECT DISTINCT
        customer_code,
        depot_code
    FROM
        mst_customer
    WHERE
        company_code='MIL' AND
        latest_record_ind=1
) CM on MESR.distributor_code = CM.customer_code
where month_date BETWEEN '2025-12-31' AND '2026-05-31' 
GROUP BY 1, 2, 3, 4
ORDER BY 1, 3, 2, 4
"""

depot_psku_primary_df = pd.read_sql(
    chain_psku_primary_query,
    prod_conn
)
depot_psku_primary_df.columns = depot_psku_primary_df.columns.str.lower()
depot_psku_primary_df['parent_material_code'] = depot_psku_primary_df['parent_material_code'].astype(int)
depot_psku_primary_df['month_date'] = pd.to_datetime(depot_psku_primary_df['month_date'])
depot_psku_primary_df = realign_pskus(depot_psku_primary_df.copy(), 'parent_material_code')
depot_psku_primary_df = depot_psku_primary_df.groupby(
    ['chain', 'parent_material_code', 'material_group_code', 'month_date'], as_index=False, dropna=False
).sum()
depot_psku_primary_df = depot_psku_primary_df[depot_psku_primary_df['parent_material_code'] != 715096]
depot_psku_primary_df.duplicated(subset=['chain', 'parent_material_code', 'month_date']).sum()
for col in ['pri_actuals_vol_rum', 'pri_apo_plan_vol_rum', 'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum']:
    if depot_psku_primary_df[col].min() < 0:
        print(col)

for col in ['pri_actuals_vol_rum', 'pri_apo_plan_vol_rum', 'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum']:
    depot_psku_primary_df[col] = depot_psku_primary_df[col].clip(lower=0)

pri_actuals_vol_rum
sec_actuals_vol_rum


In [68]:
depot_psku_primary_df[depot_psku_primary_df.duplicated(subset=['chain', 'parent_material_code', 'month_date'], keep=False)]

,chain,parent_material_code,material_group_code,month_date,pri_actuals_vol_rum,pri_apo_plan_vol_rum,sec_apo_plan_vol_rum,sec_actuals_vol_rum
962,Blinkit,732296,PABABY_ML,2025-12-31,0.00,0.000000,55.5023,0.00
963,Blinkit,732296,PABABY_ML,2026-01-31,0.00,161.650358,116.1056,0.00
964,Blinkit,732296,PABABY_SP,2025-12-31,0.00,0.000000,0.0000,33.60
965,Blinkit,732296,PABABY_SP,2026-01-31,0.00,0.000000,0.0000,0.00
970,Blinkit,732297,PABABY_ML,2025-12-31,0.00,0.000000,100.1441,0.00
971,Blinkit,732297,PABABY_ML,2026-01-31,0.00,397.753905,283.5170,0.00
972,Blinkit,732297,PABABY_ML,2026-02-28,0.00,0.000000,0.0000,0.00
973,Blinkit,732297,PABABY_SP,2025-12-31,0.00,0.000000,0.0000,137.76
974,Blinkit,732297,PABABY_SP,2026-01-31,58.22,0.000000,0.0000,58.22
975,Blinkit,732297,PABABY_SP,2026-02-28,39.36,230.724898,209.3269,39.36


In [69]:
depot_psku_primary_df = depot_psku_primary_df[depot_psku_primary_df['material_group_code']!='PABABY_SP']
depot_psku_primary_df.duplicated(subset=['chain', 'parent_material_code', 'month_date']).sum()


0

In [70]:
depot_psku_primary_df

,chain,parent_material_code,material_group_code,month_date,pri_actuals_vol_rum,pri_apo_plan_vol_rum,sec_apo_plan_vol_rum,sec_actuals_vol_rum
0,Blinkit,718288,SAFF GOLD,2025-12-31,0.000,0.000000,76.6195,66.186
1,Blinkit,718288,SAFF GOLD,2026-01-31,51.666,49.549496,52.9887,51.666
2,Blinkit,718288,SAFF GOLD,2026-02-28,61.632,55.342992,60.4012,61.632
3,Blinkit,718288,SAFF GOLD,2026-03-31,61.434,34.113398,67.0527,61.434
4,Blinkit,718288,SAFF GOLD,2026-04-30,91.068,29.985473,60.9032,91.068
...,...,...,...,...,...,...,...,...
4060,Zepto,811267,PA_ESS_HO,2026-05-31,0.000,1.067948,0.8158,0.000
4061,Zepto,811269,PA_ESS_HO,2026-05-31,0.000,2.137815,2.3543,0.000
4062,Zepto,811279,SAF_CDPRS,2026-05-31,0.000,0.512669,0.4496,0.000
4063,Zepto,811287,PA_RSW_SR,2026-04-30,26.400,114.301509,0.0000,0.000


In [71]:
depot_psku_primary_df['channel'] = 'Qcom'
depot_psku_primary_df_qcom = depot_psku_primary_df.copy()

In [72]:
depot_psku_primary_query = """
SELECT
    CASE 
        WHEN MCM.chain = 'Big basket B2C' THEN 'Big Basket'
        WHEN MCM.chain = 'Flipkart-National' THEN 'Flipkart National'
        WHEN MCM.chain = 'Flipkart-Minutes' THEN 'Flipkart National'
        WHEN MCM.chain = 'Nykaa' THEN 'Nykaa'
        WHEN MCM.chain = 'Purplle' THEN 'Purplle'
        WHEN MCM.chain = 'Myntra' THEN 'Myntra'
        WHEN MCM.chain = 'MYNTRA' THEN 'Myntra'
        WHEN MCM.chain = 'Amazon B2C' THEN 'Amazon ARIPL'
        WHEN MCM.chain = 'ARIPL' THEN 'Amazon ARIPL'
        WHEN MCM.chain = 'RK WORLDINFOCOM' THEN 'Amazon RK'
        WHEN MCM.chain = 'RKWorld' THEN 'Amazon RK'
        WHEN MCM.chain = 'Flipkart-Grocery' THEN 'Flipkart Grocery'
        WHEN MCM.chain = 'FlipkartGrocery' THEN 'Flipkart Grocery'
        WHEN MCM.chain = 'Firstcry' THEN 'First Cry'
        WHEN MCM.chain = 'CITIMALL' THEN 'City Mall'
        ELSE MCM.chain
    END AS chain,
    MM.parent_material_code,
    MM.material_group_code,
    MESR.month_date,
    SUM(pri_actuals_vol_rum) AS pri_actuals_vol_rum,
    SUM(pri_apo_plan_vol_rum) AS pri_apo_plan_vol_rum,
    SUM(MESR.sec_apo_plan_vol_rum) AS sec_apo_plan_vol_rum,
    SUM(MESR.sec_actuals_vol_rum) AS sec_actuals_vol_rum
FROM
    dwh_bpm_dist_sku_daily MESR
JOIN 
(
    SELECT
        customer,
        chain_type,
        chain
    FROM
        mst_chain_master
    WHERE
        chain_type = 'E Com B2C' AND 
        chain IN ('Flipkart-National', 'Flipkart-Grocery', 'Big basket B2C', 'RK WORLDINFOCOM', 'Amazon B2C', 
        'FATEHPURIA HYGIENE', 'Nykaa', 'Flipkart-Minutes', 'Purplle', 'Myntra', 'Dealshare', 'FlipkartGrocery', 'City Mall', '1MG', 
        'ARIPL', 'First Cry', 'Meesho', 'RKWorld', 'CITIMALL', 'Firstcry', 'EMAZING DEALS', 'MYNTRA')
) MCM ON MESR.distributor_code = MCM.customer
JOIN
(
    SELECT
        material_code,
        parent_material_code,
        material_group_code,
        uom_reporting,
        vol_per_unit
    FROM 
        mst_material
    WHERE
        company_code='MIL' AND
        latest_record_ind=1
) MM ON MESR.material_code = MM.material_code
WHERE
    MESR.month_date between '2025-12-31' and '2026-05-31'
GROUP BY 1, 2, 3, 4
ORDER BY 1, 3, 2, 4
"""
depot_psku_primary_df = pd.read_sql(
    depot_psku_primary_query,
    prod_conn
)
depot_psku_primary_df.columns = depot_psku_primary_df.columns.str.lower()
depot_psku_primary_df['parent_material_code'] = depot_psku_primary_df['parent_material_code'].astype(int)
depot_psku_primary_df['month_date'] = pd.to_datetime(depot_psku_primary_df['month_date'])
depot_psku_primary_df = realign_pskus(depot_psku_primary_df.copy(), 'parent_material_code', channel='ECOM')
depot_psku_primary_df = depot_psku_primary_df.groupby(
    ['chain', 'parent_material_code', 'material_group_code', 'month_date'], as_index=False, dropna=False
).sum()
depot_psku_primary_df = depot_psku_primary_df[depot_psku_primary_df['parent_material_code'] != 715096]
depot_psku_primary_df.duplicated(subset=['chain', 'parent_material_code', 'month_date']).sum()
for col in ['pri_actuals_vol_rum', 'pri_apo_plan_vol_rum', 'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum']:
    if depot_psku_primary_df[col].min() < 0:
        print(col)

for col in ['pri_actuals_vol_rum', 'pri_apo_plan_vol_rum', 'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum']:
    depot_psku_primary_df[col] = depot_psku_primary_df[col].clip(lower=0)

pri_actuals_vol_rum
sec_actuals_vol_rum


In [73]:
depot_psku_primary_df = depot_psku_primary_df[depot_psku_primary_df['material_group_code']!='PABABY_SP']
depot_psku_primary_df.duplicated(subset=['chain', 'parent_material_code', 'month_date']).sum()


0

In [74]:
depot_psku_primary_df['channel'] = 'Ecom'
depot_psku_primary_df_ecom = depot_psku_primary_df.copy()

In [75]:
depot_psku_primary_df = pd.concat([depot_psku_primary_df_qcom, depot_psku_primary_df_ecom], ignore_index=True)
depot_psku_primary_df

,chain,parent_material_code,material_group_code,month_date,pri_actuals_vol_rum,pri_apo_plan_vol_rum,sec_apo_plan_vol_rum,sec_actuals_vol_rum,channel
0,Blinkit,718288,SAFF GOLD,2025-12-31,0.000,0.000000,76.6195,66.186,Qcom
1,Blinkit,718288,SAFF GOLD,2026-01-31,51.666,49.549496,52.9887,51.666,Qcom
2,Blinkit,718288,SAFF GOLD,2026-02-28,61.632,55.342992,60.4012,61.632,Qcom
3,Blinkit,718288,SAFF GOLD,2026-03-31,61.434,34.113398,67.0527,61.434,Qcom
4,Blinkit,718288,SAFF GOLD,2026-04-30,91.068,29.985473,60.9032,91.068,Qcom
...,...,...,...,...,...,...,...,...,...
68920,Purplle,811279,SAF_CDPRS,2026-05-01,0.000,0.042960,0.0000,0.000,Ecom
68921,Purplle,811279,SAF_CDPRS,2026-05-31,0.000,0.000000,0.0514,0.000,Ecom
68922,Purplle,811287,PA_RSW_SR,2026-04-01,0.000,21.339603,0.0000,0.000,Ecom
68923,Purplle,811287,PA_RSW_SR,2026-05-01,0.000,5.145456,0.0000,0.000,Ecom


In [76]:
actuals_df = depot_psku_primary_df.copy()

In [77]:
actuals_df.duplicated(subset=['channel','chain', 'parent_material_code','month_date']).sum()

0

In [78]:
actuals_df = actuals_df.groupby(['channel','chain', 'parent_material_code',
       'material_group_code', 'month_date'])[['pri_actuals_vol_rum',
       'pri_apo_plan_vol_rum', 'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum']].sum().reset_index()
actuals_df

,channel,chain,parent_material_code,material_group_code,month_date,pri_actuals_vol_rum,pri_apo_plan_vol_rum,sec_apo_plan_vol_rum,sec_actuals_vol_rum
0,Ecom,1MG,718288,SAFF GOLD,2026-01-01,0.0,1.014738,0.0000,0.0
1,Ecom,1MG,718288,SAFF GOLD,2026-02-01,0.0,0.861434,0.0000,0.0
2,Ecom,1MG,718288,SAFF GOLD,2026-03-01,0.0,0.892977,0.0000,0.0
3,Ecom,1MG,718288,SAFF GOLD,2026-04-01,0.0,4.790505,0.0000,0.0
4,Ecom,1MG,718288,SAFF GOLD,2026-05-01,0.0,1.715839,0.0000,0.0
...,...,...,...,...,...,...,...,...,...
68920,Qcom,Zepto,811267,PA_ESS_HO,2026-05-31,0.0,1.067948,0.8158,0.0
68921,Qcom,Zepto,811269,PA_ESS_HO,2026-05-31,0.0,2.137815,2.3543,0.0
68922,Qcom,Zepto,811279,SAF_CDPRS,2026-05-31,0.0,0.512669,0.4496,0.0
68923,Qcom,Zepto,811287,PA_RSW_SR,2026-04-30,26.4,114.301509,0.0000,0.0


In [79]:
actuals_df.columns

Index(['channel', 'chain', 'parent_material_code', 'material_group_code',
       'month_date', 'pri_actuals_vol_rum', 'pri_apo_plan_vol_rum',
       'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum'],
      dtype='object')

In [80]:
actuals_df = actuals_df.rename(columns={
    'parent_material_code': 'psku',
    'sec_apo_plan_vol_rum': 'Consensus Vol',
    'sec_actuals_vol_rum': 'Actuals Vol',
    'month_date': 'month'
})

In [81]:
actuals_df.head()

,channel,chain,psku,material_group_code,month,pri_actuals_vol_rum,pri_apo_plan_vol_rum,Consensus Vol,Actuals Vol
0,Ecom,1MG,718288,SAFF GOLD,2026-01-01,0.0,1.014738,0.0,0.0
1,Ecom,1MG,718288,SAFF GOLD,2026-02-01,0.0,0.861434,0.0,0.0
2,Ecom,1MG,718288,SAFF GOLD,2026-03-01,0.0,0.892977,0.0,0.0
3,Ecom,1MG,718288,SAFF GOLD,2026-04-01,0.0,4.790505,0.0,0.0
4,Ecom,1MG,718288,SAFF GOLD,2026-05-01,0.0,1.715839,0.0,0.0


In [ ]:
#qcom_df.rename(columns = {'Channel':'channel'}, inplace = True)

In [82]:
qcom_df['psku'] = qcom_df['psku'].astype(int)
actuals_df['psku'] = actuals_df['psku'].astype(int)

In [83]:
actuals_df['channel'] = actuals_df['channel'].str.upper()
qcom_df['channel'] = qcom_df['channel'].str.upper()
actuals_df['chain'] = actuals_df['chain'].str.lower()
qcom_df['chain'] = qcom_df['chain'].str.lower()


In [84]:
qcom_df

,month,chain,psku,brand,portfolio,m month,run_month,calculated primary vol,channel
0,2025-12-31,blinkit,718287,PCNO(R),CNO,M,2025-12-31,0.000000,QCOM
1,2025-12-31,blinkit,718288,SAFF GOLD,Saffola Oils,M,2025-12-31,76.619500,QCOM
2,2025-12-31,blinkit,718297,PCNO(R),CNO,M,2025-12-31,0.000000,QCOM
3,2025-12-31,blinkit,718299,PCNO(R),CNO,M,2025-12-31,0.000000,QCOM
4,2025-12-31,blinkit,718300,PCNO FLEX,CNO,M,2025-12-31,0.000000,QCOM
...,...,...,...,...,...,...,...,...,...
91428,2026-08-31,zepto,810971,PA_ESS_HO,Hair Oils,M+4,2026-04-30,0.180968,QCOM
91429,2026-08-31,zepto,811005,PA_ESS_HO,Hair Oils,M+4,2026-04-30,0.154828,QCOM
91430,2026-08-31,zepto,811169,SW_SGPRF,Male Grooming,M+4,2026-04-30,0.000000,QCOM
91431,2026-08-31,zepto,811181,SAF_CDPRS,Saffola Oils,M+4,2026-04-30,0.043734,QCOM


In [ ]:
# actuals_df = actuals_df[actuals_df['month_date'] == '2025-12-31']
# actuals_df

In [85]:
x = qcom_df.copy()

In [86]:
len_before_merge = len(qcom_df)
qcom_df = qcom_df.merge(
    actuals_df.drop([ 'material_group_code', 'pri_actuals_vol_rum', 'pri_apo_plan_vol_rum'], axis=1),
    on=['channel','chain', 'psku', 'month'],
    how='left'
)
assert len_before_merge == len(qcom_df)
del len_before_merge

In [87]:
def read_qtr_ind_rate_table():
    query = """select * from DWH_SAP_INDEX_TURNOVER_MONTHWISE 
                where latest_rate_flag=1 and company_code='MIL'"""
    qtr_ind_rate_data = pd.read_sql(con=prod_conn, sql=query)
    qtr_ind_rate_data.columns = qtr_ind_rate_data.columns.str.lower()
    qtr_ind_rate =  qtr_ind_rate_data[['date', 'brand_code', 'turnover']]
    qtr_ind_rate = qtr_ind_rate.rename(columns= {'date':'month_date', 'turnover':'qtr_ind_rate'})
    
    return qtr_ind_rate

qtr_ind_rate_df = read_qtr_ind_rate_table()
qtr_ind_rate_df.head()

,month_date,brand_code,qtr_ind_rate
0,2027-03-31,PA_CN_HGO,488.152
1,2027-03-31,TRU_RAWDF,800.000
2,2027-03-31,TRU_PDRFR,850.570
3,2027-03-31,TRU_OATS,177.070
4,2027-03-31,TRU_QUINO,204.750


In [88]:
len_before_merge = len(qcom_df)
qcom_df = qcom_df.merge(
    qtr_ind_rate_df.drop('month_date', axis=1).rename(
        columns={'brand_code': 'brand', 'qtr_ind_rate': 'Index Rate'}
    ),
    on=['brand'], 
    how='left'
)
assert len_before_merge == len(qcom_df)
del len_before_merge

In [89]:
qcom_df.isna().sum()

month                         0
chain                         0
psku                          0
brand                         0
portfolio                     0
m month                       0
run_month                     0
calculated primary vol        0
channel                       0
Consensus Vol             73428
Actuals Vol               73428
Index Rate                    0
dtype: int64

In [90]:
qcom_df['Consensus Vol'] = qcom_df['Consensus Vol'].fillna(0)
qcom_df['Actuals Vol'] = qcom_df['Actuals Vol'].fillna(0)

In [91]:
qcom_df['Consensus Vol'].max()

29744.707599999998

In [92]:
qcom_df.rename(columns={'calculated primary vol': 'Stat Vol'}, inplace=True)
qcom_df

,month,chain,psku,brand,portfolio,m month,run_month,Stat Vol,channel,Consensus Vol,Actuals Vol,Index Rate
0,2025-12-31,blinkit,718287,PCNO(R),CNO,M,2025-12-31,0.000000,QCOM,0.0000,0.000,349274.001420
1,2025-12-31,blinkit,718288,SAFF GOLD,Saffola Oils,M,2025-12-31,76.619500,QCOM,76.6195,66.186,138865.260689
2,2025-12-31,blinkit,718297,PCNO(R),CNO,M,2025-12-31,0.000000,QCOM,0.0000,0.000,349274.001420
3,2025-12-31,blinkit,718299,PCNO(R),CNO,M,2025-12-31,0.000000,QCOM,0.0000,0.000,349274.001420
4,2025-12-31,blinkit,718300,PCNO FLEX,CNO,M,2025-12-31,0.000000,QCOM,0.0000,0.000,230000.000000
...,...,...,...,...,...,...,...,...,...,...,...,...
91428,2026-08-31,zepto,810971,PA_ESS_HO,Hair Oils,M+4,2026-04-30,0.180968,QCOM,0.0000,0.000,12860.631072
91429,2026-08-31,zepto,811005,PA_ESS_HO,Hair Oils,M+4,2026-04-30,0.154828,QCOM,0.0000,0.000,12860.631072
91430,2026-08-31,zepto,811169,SW_SGPRF,Male Grooming,M+4,2026-04-30,0.000000,QCOM,0.0000,0.000,1712.605337
91431,2026-08-31,zepto,811181,SAF_CDPRS,Saffola Oils,M+4,2026-04-30,0.043734,QCOM,0.0000,0.000,260000.000000


In [93]:
qcom_df['Stat Val'] = qcom_df['Stat Vol'] * qcom_df['Index Rate'] / (10 ** 7)
qcom_df['Consensus Val'] = qcom_df['Consensus Vol'] * qcom_df['Index Rate'] / (10 ** 7)
qcom_df['Actuals Val'] = qcom_df['Actuals Vol'] * qcom_df['Index Rate'] / (10 ** 7)

In [94]:
qcom_df

,month,chain,psku,brand,portfolio,m month,run_month,Stat Vol,channel,Consensus Vol,Actuals Vol,Index Rate,Stat Val,Consensus Val,Actuals Val
0,2025-12-31,blinkit,718287,PCNO(R),CNO,M,2025-12-31,0.000000,QCOM,0.0000,0.000,349274.001420,0.000000,0.000000,0.000000
1,2025-12-31,blinkit,718288,SAFF GOLD,Saffola Oils,M,2025-12-31,76.619500,QCOM,76.6195,66.186,138865.260689,1.063979,1.063979,0.919094
2,2025-12-31,blinkit,718297,PCNO(R),CNO,M,2025-12-31,0.000000,QCOM,0.0000,0.000,349274.001420,0.000000,0.000000,0.000000
3,2025-12-31,blinkit,718299,PCNO(R),CNO,M,2025-12-31,0.000000,QCOM,0.0000,0.000,349274.001420,0.000000,0.000000,0.000000
4,2025-12-31,blinkit,718300,PCNO FLEX,CNO,M,2025-12-31,0.000000,QCOM,0.0000,0.000,230000.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91428,2026-08-31,zepto,810971,PA_ESS_HO,Hair Oils,M+4,2026-04-30,0.180968,QCOM,0.0000,0.000,12860.631072,0.000233,0.000000,0.000000
91429,2026-08-31,zepto,811005,PA_ESS_HO,Hair Oils,M+4,2026-04-30,0.154828,QCOM,0.0000,0.000,12860.631072,0.000199,0.000000,0.000000
91430,2026-08-31,zepto,811169,SW_SGPRF,Male Grooming,M+4,2026-04-30,0.000000,QCOM,0.0000,0.000,1712.605337,0.000000,0.000000,0.000000
91431,2026-08-31,zepto,811181,SAF_CDPRS,Saffola Oils,M+4,2026-04-30,0.043734,QCOM,0.0000,0.000,260000.000000,0.001137,0.000000,0.000000


In [103]:
qcom_df[(qcom_df['month'] == '2026-04-30') & (qcom_df['channel']=='ECOM') & (qcom_df['run_month']=='2026-03-31')]['Actuals Val'].sum()

36.5008484674167

In [97]:
qcom_df['Stat Error'] = qcom_df['Stat Val'] - qcom_df['Actuals Val']
qcom_df['Consensus Error'] = qcom_df['Consensus Val'] - qcom_df['Actuals Val']

qcom_df['Stat Abs Error'] = np.abs(qcom_df['Stat Error'])
qcom_df['Consensus Abs Error'] = np.abs(qcom_df['Consensus Error'])

In [104]:
qcom_df['stat_bias'] = qcom_df['Stat Error']/qcom_df['Actuals Val']
qcom_df = qcom_df.fillna(0)

import numpy as np
import pandas as pd

qcom_df['stat_bias'] = (
    qcom_df['stat_bias']
    .replace([np.inf, -np.inf], 0)
    .fillna(0)
)

bins = [-np.inf, -0.15, -0.10, -0.05, 0, 0.05, 0.10, 0.15, np.inf]
labels = [
    '< -15%',
    '-15% to -10%',
    '-10% to -5%',
    '-5% to 0%',
    '0% to 5%',
    '5% to 10%',
    '10% to 15%',
    '> 15%'
]

qcom_df['stat_bias_bucket'] = pd.cut(
    qcom_df['stat_bias'],
    bins=bins,
    labels=labels,
    right=False  
)


In [ ]:
# delivery_df = pd.read_csv('/data/aman_singh/acuuracy_check/Export View of Month Vol & APO.csv')

# delivery_df['Channel'] = delivery_df['Channel'].replace({
#     'E-Commerce': 'ECOM', 'Q-Commerce': 'QCOM'})
# delivery_df
# delivery_df['delivery_vol'] = delivery_df['01-12-2025 Del Vol'].map(lambda x:0 if x.strip() == '-' else float(x.strip().replace(',','')))
# delivery_df = delivery_df[delivery_df['Channel'] == 'QCOM']
# delivery_df = delivery_df.groupby(['Depot','PSKU'])['delivery_vol'].sum().reset_index(
# )#.rename(columns = {'01-12-2025 Del Vol':'delivery_vol'})

# len_before_merge = len(qcom_df)
# qcom_df = qcom_df.merge(
#     delivery_df,
#     on=['Depot', 'PSKU'],
#     how='left'
# )
# assert len_before_merge == len(qcom_df)
# del len_before_merge

# qcom_df['Delivery Val'] = qcom_df['delivery_vol'] * qcom_df['Index Rate'] / (10 ** 7)

# qcom_df['Dp Error'] = qcom_df['Delivery Val'] - qcom_df['Actuals Val']

# qcom_df['Dp Abs Error'] = np.abs(qcom_df['Dp Error'])



In [105]:
qcom_df.columns = qcom_df.columns.str.lower()
qcom_df

,month,chain,psku,brand,portfolio,m month,run_month,stat vol,channel,consensus vol,...,index rate,stat val,consensus val,actuals val,stat error,consensus error,stat abs error,consensus abs error,stat_bias,stat_bias_bucket
0,2025-12-31,blinkit,718287,PCNO(R),CNO,M,2025-12-31,0.000000,QCOM,0.0000,...,349274.001420,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%
1,2025-12-31,blinkit,718288,SAFF GOLD,Saffola Oils,M,2025-12-31,76.619500,QCOM,76.6195,...,138865.260689,1.063979,1.063979,0.919094,0.144885,0.144885,0.144885,0.144885,0.157639,> 15%
2,2025-12-31,blinkit,718297,PCNO(R),CNO,M,2025-12-31,0.000000,QCOM,0.0000,...,349274.001420,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%
3,2025-12-31,blinkit,718299,PCNO(R),CNO,M,2025-12-31,0.000000,QCOM,0.0000,...,349274.001420,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%
4,2025-12-31,blinkit,718300,PCNO FLEX,CNO,M,2025-12-31,0.000000,QCOM,0.0000,...,230000.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91428,2026-08-31,zepto,810971,PA_ESS_HO,Hair Oils,M+4,2026-04-30,0.180968,QCOM,0.0000,...,12860.631072,0.000233,0.000000,0.000000,0.000233,0.000000,0.000233,0.000000,0.000000,0% to 5%
91429,2026-08-31,zepto,811005,PA_ESS_HO,Hair Oils,M+4,2026-04-30,0.154828,QCOM,0.0000,...,12860.631072,0.000199,0.000000,0.000000,0.000199,0.000000,0.000199,0.000000,0.000000,0% to 5%
91430,2026-08-31,zepto,811169,SW_SGPRF,Male Grooming,M+4,2026-04-30,0.000000,QCOM,0.0000,...,1712.605337,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%
91431,2026-08-31,zepto,811181,SAF_CDPRS,Saffola Oils,M+4,2026-04-30,0.043734,QCOM,0.0000,...,260000.000000,0.001137,0.000000,0.000000,0.001137,0.000000,0.001137,0.000000,0.000000,0% to 5%


In [ ]:
# df = pd.read_excel("/data/aman_singh/acuuracy_check/acc_framework_feb.xlsx")
# df.columns = df.columns.str.lower()
# df = df[df['forecast_granularity']!='Depot x PSKU']

In [ ]:
df.columns = df.columns.str.lower()

In [ ]:
qcom_df.columns

Index(['month', 'depot', 'psku', 'brand', 'portfolio', 'm month', 'run_month',
       'stat vol', 'channel', 'consensus vol', 'actuals vol', 'index rate',
       'stat val', 'consensus val', 'actuals val', 'stat error',
       'consensus error', 'stat abs error', 'consensus abs error', 'stat_bias',
       'stat_bias_bucket'],
      dtype='object')

In [106]:

qcom_df['forecast_granularity'] = 'Chain x PSKU'
qcom_df['forecast_type'] = 'offtakes_to_primary'
#qcom_df['channel'] = 'QCOM'
qcom_df

,month,chain,psku,brand,portfolio,m month,run_month,stat vol,channel,consensus vol,...,consensus val,actuals val,stat error,consensus error,stat abs error,consensus abs error,stat_bias,stat_bias_bucket,forecast_granularity,forecast_type
0,2025-12-31,blinkit,718287,PCNO(R),CNO,M,2025-12-31,0.000000,QCOM,0.0000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%,Chain x PSKU,offtakes_to_primary
1,2025-12-31,blinkit,718288,SAFF GOLD,Saffola Oils,M,2025-12-31,76.619500,QCOM,76.6195,...,1.063979,0.919094,0.144885,0.144885,0.144885,0.144885,0.157639,> 15%,Chain x PSKU,offtakes_to_primary
2,2025-12-31,blinkit,718297,PCNO(R),CNO,M,2025-12-31,0.000000,QCOM,0.0000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%,Chain x PSKU,offtakes_to_primary
3,2025-12-31,blinkit,718299,PCNO(R),CNO,M,2025-12-31,0.000000,QCOM,0.0000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%,Chain x PSKU,offtakes_to_primary
4,2025-12-31,blinkit,718300,PCNO FLEX,CNO,M,2025-12-31,0.000000,QCOM,0.0000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%,Chain x PSKU,offtakes_to_primary
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91428,2026-08-31,zepto,810971,PA_ESS_HO,Hair Oils,M+4,2026-04-30,0.180968,QCOM,0.0000,...,0.000000,0.000000,0.000233,0.000000,0.000233,0.000000,0.000000,0% to 5%,Chain x PSKU,offtakes_to_primary
91429,2026-08-31,zepto,811005,PA_ESS_HO,Hair Oils,M+4,2026-04-30,0.154828,QCOM,0.0000,...,0.000000,0.000000,0.000199,0.000000,0.000199,0.000000,0.000000,0% to 5%,Chain x PSKU,offtakes_to_primary
91430,2026-08-31,zepto,811169,SW_SGPRF,Male Grooming,M+4,2026-04-30,0.000000,QCOM,0.0000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%,Chain x PSKU,offtakes_to_primary
91431,2026-08-31,zepto,811181,SAF_CDPRS,Saffola Oils,M+4,2026-04-30,0.043734,QCOM,0.0000,...,0.000000,0.000000,0.001137,0.000000,0.001137,0.000000,0.000000,0% to 5%,Chain x PSKU,offtakes_to_primary


In [107]:
qcom_df = qcom_df[qcom_df['m month'] == 'M+1']

In [109]:
qcom_df.to_csv('/data/aman_singh/acuuracy_check/chain_psku_primary_accuracy_till_may.csv')

In [1]:
import pandas as pd
df = pd.read_excel('/data/aman_singh/acuuracy_check/acc_framework_may_final.xlsx', sheet_name = 'Sheet1')
df

,Unnamed: 0,forecast_granularity,forecast_type,channel,portfolio,brand,run_month,m month,month,depot,...,stat error,consensus error,stat abs error,consensus abs error,prophet vol_value,rf_vol_value,prophet heuristic vol_value,rf heuristic vol_value,stat_bias,stat_bias_bucket
0,129118,Depot x PSKU,secondary,GT,CNO,KERALA,2026-04-30,M+1,2026-05-31,D673,...,0.022506,0.041685,0.022506,0.041685,0.0,0.0,0.0,0.0,0.201333,> 15%
1,129119,Depot x PSKU,secondary,GT,CNO,KERALA,2026-04-30,M+1,2026-05-31,D673,...,-0.004181,-0.002461,0.004181,0.002461,0.0,0.0,0.0,0.0,-0.132249,-15% to -10%
2,129120,Depot x PSKU,secondary,GT,CNO,KERALA,2026-04-30,M+1,2026-05-31,D674,...,0.003741,-0.002501,0.003741,0.002501,0.0,0.0,0.0,0.0,0.334012,> 15%
3,129121,Depot x PSKU,secondary,GT,CNO,KERALA,2026-04-30,M+1,2026-05-31,D674,...,-0.020124,0.019797,0.020124,0.019797,0.0,0.0,0.0,0.0,-0.223653,< -15%
4,129122,Depot x PSKU,secondary,GT,CNO,KERALA,2026-04-30,M+1,2026-05-31,D676,...,0.074692,0.070734,0.074692,0.070734,0.0,0.0,0.0,0.0,0.218949,> 15%
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
38287,451102,Depot x PSKU,offtakes_to_primary,ECOM,Male Grooming,SW_SGPRF,2026-04-30,M+1,2026-05-31,D677,...,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,0.000000,0% to 5%
38288,451103,Depot x PSKU,offtakes_to_primary,ECOM,Saffola Oils,SAF_CDPRS,2026-04-30,M+1,2026-05-31,D677,...,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,0.000000,0% to 5%
38289,451104,Depot x PSKU,offtakes_to_primary,ECOM,Hair Oils,PA_ESS_HO,2026-04-30,M+1,2026-05-31,D677,...,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,0.000000,0% to 5%
38290,451105,Depot x PSKU,offtakes_to_primary,ECOM,Hair Oils,PA_ESS_HO,2026-04-30,M+1,2026-05-31,D677,...,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,0.000000,0% to 5%


In [2]:
qcom_df = pd.read_csv('/data/aman_singh/acuuracy_check/chain_psku_primary_accuracy_till_may.csv')
qcom_df

,Unnamed: 0,month,chain,psku,brand,portfolio,m month,run_month,stat vol,channel,...,consensus val,actuals val,stat error,consensus error,stat abs error,consensus abs error,stat_bias,stat_bias_bucket,forecast_granularity,forecast_type
0,1571,2026-01-31,blinkit,718287,PCNO(R),CNO,M+1,2025-12-31,0.000000,QCOM,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%,Chain x PSKU,offtakes_to_primary
1,1572,2026-01-31,blinkit,718288,SAFF GOLD,Saffola Oils,M+1,2025-12-31,52.041436,QCOM,...,0.735829,0.717461,0.005213,0.018368,0.005213,0.018368,0.007267,0% to 5%,Chain x PSKU,offtakes_to_primary
2,1573,2026-01-31,blinkit,718297,PCNO(R),CNO,M+1,2025-12-31,0.000000,QCOM,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%,Chain x PSKU,offtakes_to_primary
3,1574,2026-01-31,blinkit,718299,PCNO(R),CNO,M+1,2025-12-31,0.000000,QCOM,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%,Chain x PSKU,offtakes_to_primary
4,1575,2026-01-31,blinkit,718300,PCNO FLEX,CNO,M+1,2025-12-31,0.000000,QCOM,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%,Chain x PSKU,offtakes_to_primary
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24588,86523,2026-05-31,zepto,810971,PA_ESS_HO,Hair Oils,M+1,2026-04-30,0.180968,QCOM,...,0.000102,0.011112,-0.010879,-0.011010,0.010879,0.011010,-0.979055,< -15%,Chain x PSKU,offtakes_to_primary
24589,86524,2026-05-31,zepto,811005,PA_ESS_HO,Hair Oils,M+1,2026-04-30,0.154828,QCOM,...,0.007044,0.003889,-0.003690,0.003155,0.003690,0.003155,-0.948800,< -15%,Chain x PSKU,offtakes_to_primary
24590,86525,2026-05-31,zepto,811169,SW_SGPRF,Male Grooming,M+1,2026-04-30,0.000000,QCOM,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%,Chain x PSKU,offtakes_to_primary
24591,86526,2026-05-31,zepto,811181,SAF_CDPRS,Saffola Oils,M+1,2026-04-30,0.043734,QCOM,...,0.046896,0.000000,0.001137,0.046896,0.001137,0.046896,0.000000,0% to 5%,Chain x PSKU,offtakes_to_primary


In [15]:
offtakes_df = pd.read_csv('/data/aman_singh/acuuracy_check/acc_offtakes_till_may.csv')
offtakes_df

,Unnamed: 0,month,platform_name,parent_material_code,brand,portfolio,run_month,m month,pred_prophet,pred_rf,...,vol_in_rum,Index Rate,offtakes_forecasted_vol,offtakes actuals val,offtakes error,offtakes abs error,offtakes_bias,offtakes_bias_bucket,forecast_granularity,forecast_type
0,0,2026-05-31,amazon aripl,718288,SAFF GOLD,Saffola Oils,2026-04-30,M+1,29.403313,27.76920,...,30.2040,138865.260689,30.318787,0.419429,-0.001594,0.001594,-0.003800,-5% to 0%,Chain x PSKU,offtakes
1,1,2026-05-31,amazon aripl,718321,SAFF KO,Saffola Oils,2026-04-30,M+1,0.000000,0.00000,...,0.0000,168827.536176,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%,Chain x PSKU,offtakes
2,2,2026-05-31,amazon aripl,718322,SAFF KO,Saffola Oils,2026-04-30,M+1,7.271279,16.05050,...,12.1500,168827.536176,11.112333,0.205125,0.017519,0.017519,0.085405,5% to 10%,Chain x PSKU,offtakes
3,3,2026-05-31,amazon aripl,718323,SF_IMV_MK,Foods,2026-04-30,M+1,0.000000,0.00000,...,0.0000,739265.904412,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%,Chain x PSKU,offtakes
4,4,2026-05-31,amazon aripl,718328,SAFF KOCO,Saffola Oils,2026-04-30,M+1,3.942144,6.99084,...,9.6561,123636.889888,9.217630,0.119385,0.005421,0.005421,0.045409,0% to 5%,Chain x PSKU,offtakes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16426,16426,2026-04-30,nykaa,810605,KAYA_ML,Skin Care,2026-03-31,M+1,0.000000,0.00000,...,0.0000,1226.374229,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%,Chain x PSKU,offtakes
16427,16427,2026-04-30,nykaa,810673,PA_ESS_HO,Hair Oils,2026-03-31,M+1,0.000000,0.00000,...,1.7640,12860.631072,1.844295,0.002269,-0.000103,0.000103,-0.045519,-5% to 0%,Chain x PSKU,offtakes
16428,16428,2026-04-30,nykaa,810674,PA_ESS_HO,Hair Oils,2026-03-31,M+1,0.000000,0.00000,...,0.5880,12860.631072,0.407243,0.000756,0.000232,0.000232,0.307410,> 15%,Chain x PSKU,offtakes
16429,16429,2026-04-30,nykaa,810738,PABABY_GM,Skin Care,2026-03-31,M+1,0.000000,0.00000,...,0.0000,366.484998,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%,Chain x PSKU,offtakes


In [16]:
offtakes_df.rename(columns={'platform_name':'chain', 'parent_material_code':'psku'
                            ,'Index Rate':'index rate'}, inplace=True)
offtakes_df.drop(['vol_in_rum','run_month'], axis=1, inplace=True)
offtakes_df

,Unnamed: 0,month,chain,psku,brand,portfolio,m month,pred_prophet,pred_rf,offtakes_forecasted_value,channel,index rate,offtakes_forecasted_vol,offtakes actuals val,offtakes error,offtakes abs error,offtakes_bias,offtakes_bias_bucket,forecast_granularity,forecast_type
0,0,2026-05-31,amazon aripl,718288,SAFF GOLD,Saffola Oils,M+1,29.403313,27.76920,0.421023,ECOM,138865.260689,30.318787,0.419429,-0.001594,0.001594,-0.003800,-5% to 0%,Chain x PSKU,offtakes
1,1,2026-05-31,amazon aripl,718321,SAFF KO,Saffola Oils,M+1,0.000000,0.00000,0.000000,ECOM,168827.536176,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%,Chain x PSKU,offtakes
2,2,2026-05-31,amazon aripl,718322,SAFF KO,Saffola Oils,M+1,7.271279,16.05050,0.187607,ECOM,168827.536176,11.112333,0.205125,0.017519,0.017519,0.085405,5% to 10%,Chain x PSKU,offtakes
3,3,2026-05-31,amazon aripl,718323,SF_IMV_MK,Foods,M+1,0.000000,0.00000,0.000000,ECOM,739265.904412,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%,Chain x PSKU,offtakes
4,4,2026-05-31,amazon aripl,718328,SAFF KOCO,Saffola Oils,M+1,3.942144,6.99084,0.113964,ECOM,123636.889888,9.217630,0.119385,0.005421,0.005421,0.045409,0% to 5%,Chain x PSKU,offtakes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16426,16426,2026-04-30,nykaa,810605,KAYA_ML,Skin Care,M+1,0.000000,0.00000,0.000000,ECOM,1226.374229,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%,Chain x PSKU,offtakes
16427,16427,2026-04-30,nykaa,810673,PA_ESS_HO,Hair Oils,M+1,0.000000,0.00000,0.002372,ECOM,12860.631072,1.844295,0.002269,-0.000103,0.000103,-0.045519,-5% to 0%,Chain x PSKU,offtakes
16428,16428,2026-04-30,nykaa,810674,PA_ESS_HO,Hair Oils,M+1,0.000000,0.00000,0.000524,ECOM,12860.631072,0.407243,0.000756,0.000232,0.000232,0.307410,> 15%,Chain x PSKU,offtakes
16429,16429,2026-04-30,nykaa,810738,PABABY_GM,Skin Care,M+1,0.000000,0.00000,0.000000,ECOM,366.484998,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%,Chain x PSKU,offtakes


In [20]:
offtakes_df['month'].unique()
offtakes_df = offtakes_df[offtakes_df['month'].isin(['2026-04-30', '2026-05-31'])]


In [21]:
final_df = pd.concat([df,qcom_df])
final_df

,Unnamed: 0,forecast_granularity,forecast_type,channel,portfolio,brand,run_month,m month,month,depot,...,consensus error,stat abs error,consensus abs error,prophet vol_value,rf_vol_value,prophet heuristic vol_value,rf heuristic vol_value,stat_bias,stat_bias_bucket,chain
0,129118,Depot x PSKU,secondary,GT,CNO,KERALA,2026-04-30 00:00:00,M+1,2026-05-31 00:00:00,D673,...,0.041685,0.022506,0.041685,0.0,0.0,0.0,0.0,0.201333,> 15%,NaN
1,129119,Depot x PSKU,secondary,GT,CNO,KERALA,2026-04-30 00:00:00,M+1,2026-05-31 00:00:00,D673,...,-0.002461,0.004181,0.002461,0.0,0.0,0.0,0.0,-0.132249,-15% to -10%,NaN
2,129120,Depot x PSKU,secondary,GT,CNO,KERALA,2026-04-30 00:00:00,M+1,2026-05-31 00:00:00,D674,...,-0.002501,0.003741,0.002501,0.0,0.0,0.0,0.0,0.334012,> 15%,NaN
3,129121,Depot x PSKU,secondary,GT,CNO,KERALA,2026-04-30 00:00:00,M+1,2026-05-31 00:00:00,D674,...,0.019797,0.020124,0.019797,0.0,0.0,0.0,0.0,-0.223653,< -15%,NaN
4,129122,Depot x PSKU,secondary,GT,CNO,KERALA,2026-04-30 00:00:00,M+1,2026-05-31 00:00:00,D676,...,0.070734,0.074692,0.070734,0.0,0.0,0.0,0.0,0.218949,> 15%,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24588,86523,Chain x PSKU,offtakes_to_primary,QCOM,Hair Oils,PA_ESS_HO,2026-04-30,M+1,2026-05-31,NaN,...,-0.011010,0.010879,0.011010,NaN,NaN,NaN,NaN,-0.979055,< -15%,zepto
24589,86524,Chain x PSKU,offtakes_to_primary,QCOM,Hair Oils,PA_ESS_HO,2026-04-30,M+1,2026-05-31,NaN,...,0.003155,0.003690,0.003155,NaN,NaN,NaN,NaN,-0.948800,< -15%,zepto
24590,86525,Chain x PSKU,offtakes_to_primary,QCOM,Male Grooming,SW_SGPRF,2026-04-30,M+1,2026-05-31,NaN,...,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,0.000000,0% to 5%,zepto
24591,86526,Chain x PSKU,offtakes_to_primary,QCOM,Saffola Oils,SAF_CDPRS,2026-04-30,M+1,2026-05-31,NaN,...,0.046896,0.001137,0.046896,NaN,NaN,NaN,NaN,0.000000,0% to 5%,zepto


In [11]:
x = final_df.copy()

In [22]:
offtakes_df

,Unnamed: 0,month,chain,psku,brand,portfolio,m month,pred_prophet,pred_rf,offtakes_forecasted_value,channel,index rate,offtakes_forecasted_vol,offtakes actuals val,offtakes error,offtakes abs error,offtakes_bias,offtakes_bias_bucket,forecast_granularity,forecast_type
0,0,2026-05-31,amazon aripl,718288,SAFF GOLD,Saffola Oils,M+1,29.403313,27.76920,0.421023,ECOM,138865.260689,30.318787,0.419429,-0.001594,0.001594,-0.003800,-5% to 0%,Chain x PSKU,offtakes
1,1,2026-05-31,amazon aripl,718321,SAFF KO,Saffola Oils,M+1,0.000000,0.00000,0.000000,ECOM,168827.536176,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%,Chain x PSKU,offtakes
2,2,2026-05-31,amazon aripl,718322,SAFF KO,Saffola Oils,M+1,7.271279,16.05050,0.187607,ECOM,168827.536176,11.112333,0.205125,0.017519,0.017519,0.085405,5% to 10%,Chain x PSKU,offtakes
3,3,2026-05-31,amazon aripl,718323,SF_IMV_MK,Foods,M+1,0.000000,0.00000,0.000000,ECOM,739265.904412,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%,Chain x PSKU,offtakes
4,4,2026-05-31,amazon aripl,718328,SAFF KOCO,Saffola Oils,M+1,3.942144,6.99084,0.113964,ECOM,123636.889888,9.217630,0.119385,0.005421,0.005421,0.045409,0% to 5%,Chain x PSKU,offtakes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16426,16426,2026-04-30,nykaa,810605,KAYA_ML,Skin Care,M+1,0.000000,0.00000,0.000000,ECOM,1226.374229,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%,Chain x PSKU,offtakes
16427,16427,2026-04-30,nykaa,810673,PA_ESS_HO,Hair Oils,M+1,0.000000,0.00000,0.002372,ECOM,12860.631072,1.844295,0.002269,-0.000103,0.000103,-0.045519,-5% to 0%,Chain x PSKU,offtakes
16428,16428,2026-04-30,nykaa,810674,PA_ESS_HO,Hair Oils,M+1,0.000000,0.00000,0.000524,ECOM,12860.631072,0.407243,0.000756,0.000232,0.000232,0.307410,> 15%,Chain x PSKU,offtakes
16429,16429,2026-04-30,nykaa,810738,PABABY_GM,Skin Care,M+1,0.000000,0.00000,0.000000,ECOM,366.484998,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%,Chain x PSKU,offtakes


In [25]:
# final_df = pd.concat([final_df,offtakes_df])
#final_df
final_df.to_csv('acc_till_may.csv')

In [24]:
final_df.isnull().sum()

Unnamed: 0                         0
forecast_granularity               0
forecast_type                      0
channel                            0
portfolio                          0
brand                              0
run_month                       8129
m month                            0
month                              0
depot                          32722
psku                               0
stat vol                        8129
prophet vol                    61110
rf_vol                         61110
prophet heuristic vol          61110
rf heuristic vol               61110
consensus vol                   8129
actuals vol                     8129
key                            61110
index rate                         0
stat val                        8129
consensus val                   8129
actuals val                     8129
stat error                      8129
consensus error                 8129
stat abs error                  8129
consensus abs error             8129
p

In [7]:
import pickle
with open('/data/aman_singh/acuuracy_check/prophet_models (2).pkl', 'rb') as f:
    model = pickle.load(f)

In [6]:
model['ORS_D535_731588']

{'changepoint_prior_scale': 0.01,
 'changepoint_range': 0.8,
 'seasonality_prior_scale': 2.0,
 'n_changepoints': 4,
 'yearly_seasonality': 5}

In [8]:
model['ORS_D535_731588']

{'changepoint_prior_scale': 0.01,
 'changepoint_range': 0.8,
 'seasonality_prior_scale': 0.1,
 'n_changepoints': 4,
 'yearly_seasonality': 4}

In [2]:
print(model.growth)

AttributeError: 'dict' object has no attribute 'growth'